# Exploração da Camada Bronze — Health Insurance Marketplace

Este notebook serve como ponto de partida para a **exploração dos dados brutos** ingeridos na camada Bronze do Data Lake.

## Objetivo

- Listar todas as tabelas do database `eedb015_bronze` no AWS Glue Catalog.
- Para cada tabela, exibir as primeiras 10 linhas via **Amazon Athena** (usando `awswrangler`).
- Entender a estrutura e o conteúdo de cada tabela para guiar análises futuras nas camadas Silver e Gold.

## Como executar

1. Abra o projeto no Dev Container (`Ctrl+Shift+P` → **Dev Containers: Reopen in Container**).
2. Atualize as credenciais no arquivo `.env` e depois rode a task do VsCode **Refresh AWS Credentials**, se a sessão do Learner Lab tiver expirado.
3. Selecione o kernel **Python 3.11.14**.
4. Execute as células sequencialmente.

> **Atenção:** este notebook usa `awswrangler` para consultar o Athena — diferente do `landing_to_bronze.ipynb` que usa PySpark/Glue. Não é necessário inicializar SparkContext aqui.

---
## 1. Configuração e Imports

In [1]:
import awswrangler as wr
import boto3
import pandas as pd
from IPython.display import Markdown, display

# Exibe todas as colunas e evita truncamento
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.width", 300)

In [2]:
# Descobre o Account ID dinamicamente — funciona para qualquer membro do grupo
account_id = boto3.client("sts").get_caller_identity()["Account"]

BRONZE_DATABASE   = "eedb015_bronze"
S3_ATHENA_OUTPUT  = f"s3://{account_id}-eedb015-athena-results/"

print(f"Account ID      : {account_id}")
print(f"Bronze Database : {BRONZE_DATABASE}")
print(f"Athena Output   : {S3_ATHENA_OUTPUT}")

Account ID      : 637423524537
Bronze Database : eedb015_bronze
Athena Output   : s3://637423524537-eedb015-athena-results/


---
## 2. Descoberta de Tabelas

In [3]:
# Lista todas as tabelas do database Bronze no Glue Catalog
tables_df = wr.catalog.tables(database=BRONZE_DATABASE)

print(f"Total de tabelas no database '{BRONZE_DATABASE}': {len(tables_df)}")
display(tables_df[["Table", "TableType"]].sort_values("Table").reset_index(drop=True))

Total de tabelas no database 'eedb015_bronze': 42


,Table,TableType
0,benefits_cost_sharing,EXTERNAL_TABLE
1,business_rules,EXTERNAL_TABLE
2,crosswalk2015,EXTERNAL_TABLE
3,crosswalk2016,EXTERNAL_TABLE
4,network,EXTERNAL_TABLE
5,plan_attributes,EXTERNAL_TABLE
6,rate,EXTERNAL_TABLE
7,raw_2014_benefits_cost_sharing_puf,EXTERNAL_TABLE
8,raw_2014_business_rules_puf,EXTERNAL_TABLE
9,raw_2014_network_puf,EXTERNAL_TABLE


In [4]:
# Funções auxiliares usadas nas seções seguintes

all_table_names: list[str] = sorted(tables_df["Table"].tolist())


def tables_by_pattern(*patterns: str) -> list[str]:
    """Retorna tabelas cujo nome contenha qualquer um dos padrões fornecidos."""
    result = []
    for t in all_table_names:
        if any(p in t for p in patterns):
            result.append(t)
    return result


def preview_table(table_name: str, limit: int = 10) -> pd.DataFrame:
    """Retorna as primeiras `limit` linhas de uma tabela via Athena."""
    sql = f'SELECT * FROM "{BRONZE_DATABASE}"."{table_name}" LIMIT {limit}'
    try:
        df = wr.athena.read_sql_query(
            sql=sql,
            database=BRONZE_DATABASE,
            s3_output=S3_ATHENA_OUTPUT,
            ctas_approach=False,
        )
        return df
    except Exception as exc:
        print(f"[ERRO] Não foi possível consultar '{table_name}': {exc}")
        return pd.DataFrame()


def show_family(patterns: list[str]) -> None:
    """Exibe preview de todas as tabelas que casam com os padrões informados."""
    matched = tables_by_pattern(*patterns)
    if not matched:
        print("Nenhuma tabela encontrada para este padrão.")
        return
    for table in matched:
        display(Markdown(f"### `{table}`"))
        df = preview_table(table)
        if not df.empty:
            display(df)
            print(f"   ↳ {len(df.columns)} colunas | {len(df)} linhas exibidas")
        print()

---
## 3. Tabela: Rate — Tarifas dos Planos

**O que é:** Contém os **prêmios mensais** cobrados por cada plano de saúde, segmentados por faixa etária, perfil de tabaco e área geográfica de tarifação. É a **maior tabela do dataset** — aproximadamente 50 milhões de linhas no período 2014–2016, pois cada plano gera ~50–80 linhas (uma por combinação de idade × área geográfica × perfil de tabaco).

**Colunas principais:**

| Coluna | Tipo | Descrição |
|---|---|---|
| `BusinessYear` | integer | Ano do plano (2014, 2015 ou 2016) |
| `StateCode` | string | Estado norte-americano (2 letras, ex: `CA`, `TX`) |
| `IssuerId` | string | Código da seguradora emissora |
| `SourceName` | string | Fonte/exchange (ex: `HIOS`, `SBM`) |
| `VersionNum` | integer | Número de versão da submissão |
| `ImportDate` | date | Data de importação do arquivo |
| `IssuerId2` | string | Identificador alternativo da seguradora |
| `FederalTIN` | string | Federal Tax Identification Number |
| `RateEffectiveDate` | date | Início de vigência da tarifa |
| `RateExpirationDate` | date | Término de vigência da tarifa |
| `PlanId` | string | ID do plano no formato HIOS — chave de ligação com `PlanAttributes` |
| `RatingAreaId` | string | ID da área de tarifação (sub-região geográfica) |
| `Tobacco` | string | Perfil de tabaco: `Tobacco User` / `No Preference` |
| `Age` | string | Faixa etária: `0-20`, `21`, `27`, `30`, ..., `64`, `Family Option` |
| `IndividualRate` | decimal | Prêmio mensal individual em USD |
| `IndividualTobaccoRate` | decimal | Prêmio para fumante individual |
| `Couple` | decimal | Prêmio para casal |
| `PrimarySubscriberAndOneDependent` | decimal | Titular + 1 dependente |
| `PrimarySubscriberAndTwoDependents` | decimal | Titular + 2 dependentes |
| `PrimarySubscriberAndThreeOrMoreDependents` | decimal | Titular + 3+ dependentes |
| `CoupleAndOneDependent` | decimal | Casal + 1 dependente |
| `CoupleAndTwoDependents` | decimal | Casal + 2 dependentes |
| `CoupleAndThreeOrMoreDependents` | decimal | Casal + 3+ dependentes |

**Relações:** `PlanId` → `PlanAttributes.PlanId`; `IssuerId` + `StateCode` → `ServiceArea`

**Insights dos notebooks Kaggle:**
- Prêmios para 64 anos chegam a ser **3× o prêmio de 21 anos** no mesmo plano (limite máximo permitido pelo ACA).
- A variável `Tobacco` adiciona surcharge de ~50% sobre o prêmio base (teto definido em lei).
- Estados com maior competição (10+ seguradoras) apresentam prêmios médios menores — correlação negativa especialmente no nível `Silver`.

**Relevância para o projeto:** Responde diretamente às questões 2 (competição × preço), 3 (precificação) e 4 (rede × preço). É a tabela de fato principal para análises de valor.

> ⚠️ **Nota de escala:** Por ser a maior tabela (~50M linhas), consultas no Athena sem filtro de `StateCode` ou `BusinessYear` são lentas e custosas — sempre filtre antes de agregar.

In [5]:
show_family(["rate"])

### `rate`

,businessyear,statecode,issuerid,sourcename,versionnum,importdate,issuerid2,federaltin,rateeffectivedate,rateexpirationdate,planid,ratingareaid,tobacco,age,individualrate,individualtobaccorate,couple,primarysubscriberandonedependent,primarysubscriberandtwodependents,primarysubscriberandthreeormoredependents,coupleandonedependent,coupleandtwodependents,coupleandthreeormoredependents,rownumber,landinzone_path,ingestion_datetime
0,2014,AK,21989,HIOS,6,2014-03-19 07:06:49,21989,93-0438772,2014-01-01,2014-12-31,21989AK0010001,Rating Area 1,No Preference,0-20,29.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,14,s3://637423524537-eedb015-landing/raw/health_insurance/Rate.csv,2026-04-04 15:49:20.674256+00:00
1,2014,AK,21989,HIOS,6,2014-03-19 07:06:49,21989,93-0438772,2014-01-01,2014-12-31,21989AK0020001,Rating Area 1,No Preference,Family Option,36.95,<NA>,73.9,107.61,107.61,107.61,144.56,144.56,144.56,14,s3://637423524537-eedb015-landing/raw/health_insurance/Rate.csv,2026-04-04 15:49:20.674256+00:00
2,2014,AK,21989,HIOS,6,2014-03-19 07:06:49,21989,93-0438772,2014-01-01,2014-12-31,21989AK0020001,Rating Area 2,No Preference,Family Option,36.95,<NA>,73.9,107.61,107.61,107.61,144.56,144.56,144.56,15,s3://637423524537-eedb015-landing/raw/health_insurance/Rate.csv,2026-04-04 15:49:20.674256+00:00
3,2014,AK,21989,HIOS,6,2014-03-19 07:06:49,21989,93-0438772,2014-01-01,2014-12-31,21989AK0010001,Rating Area 1,No Preference,21,32.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,15,s3://637423524537-eedb015-landing/raw/health_insurance/Rate.csv,2026-04-04 15:49:20.674256+00:00
4,2014,AK,21989,HIOS,6,2014-03-19 07:06:49,21989,93-0438772,2014-01-01,2014-12-31,21989AK0010001,Rating Area 1,No Preference,22,32.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,16,s3://637423524537-eedb015-landing/raw/health_insurance/Rate.csv,2026-04-04 15:49:20.674256+00:00
5,2014,AK,21989,HIOS,6,2014-03-19 07:06:49,21989,93-0438772,2014-01-01,2014-12-31,21989AK0020001,Rating Area 3,No Preference,Family Option,36.95,<NA>,73.9,107.61,107.61,107.61,144.56,144.56,144.56,16,s3://637423524537-eedb015-landing/raw/health_insurance/Rate.csv,2026-04-04 15:49:20.674256+00:00
6,2014,AK,21989,HIOS,6,2014-03-19 07:06:49,21989,93-0438772,2014-01-01,2014-12-31,21989AK0020002,Rating Area 1,No Preference,Family Option,32.45,<NA>,64.9,94.5,94.5,94.5,126.95,126.95,126.95,17,s3://637423524537-eedb015-landing/raw/health_insurance/Rate.csv,2026-04-04 15:49:20.674256+00:00
7,2014,AK,21989,HIOS,6,2014-03-19 07:06:49,21989,93-0438772,2014-01-01,2014-12-31,21989AK0010001,Rating Area 1,No Preference,23,32.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,17,s3://637423524537-eedb015-landing/raw/health_insurance/Rate.csv,2026-04-04 15:49:20.674256+00:00
8,2014,AK,21989,HIOS,6,2014-03-19 07:06:49,21989,93-0438772,2014-01-01,2014-12-31,21989AK0010001,Rating Area 1,No Preference,24,32.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,18,s3://637423524537-eedb015-landing/raw/health_insurance/Rate.csv,2026-04-04 15:49:20.674256+00:00
9,2014,AK,21989,HIOS,6,2014-03-19 07:06:49,21989,93-0438772,2014-01-01,2014-12-31,21989AK0020002,Rating Area 2,No Preference,Family Option,32.45,<NA>,64.9,94.5,94.5,94.5,126.95,126.95,126.95,18,s3://637423524537-eedb015-landing/raw/health_insurance/Rate.csv,2026-04-04 15:49:20.674256+00:00


   ↳ 26 colunas | 10 linhas exibidas



### `raw_2014_rate_puf`

,businessyear,statecode,issuerid,sourcename,versionnum,importdate,issuerid2,federaltin,rateeffectivedate,rateexpirationdate,planid,ratingareaid,tobacco,age,individualrate,individualtobaccorate,couple,primarysubscriberandonedependent,primarysubscriberandtwodependents,primarysubscriberandthreeormoredependents,coupleandonedependent,coupleandtwodependents,coupleandthreeormoredependents,rownumber,landinzone_path,ingestion_datetime
0,2014,AK,21989,HIOS,6,2014-03-19 07:06:49,21989,93-0438772,2014-01-01,2014-12-31,21989AK0010001,Rating Area 1,No Preference,0-20,29.00,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,14,s3://637423524537-eedb015-landing/raw/health_insurance/raw/2014/Rate_PUF.csv,2026-04-04 15:50:22.777322+00:00
1,2014,AK,21989,HIOS,6,2014-03-19 07:06:49,21989,93-0438772,2014-01-01,2014-12-31,21989AK0020001,Rating Area 1,No Preference,Family Option,36.95,<NA>,73.90,107.61,107.61,107.61,144.56,144.56,144.56,14,s3://637423524537-eedb015-landing/raw/health_insurance/raw/2014/Rate_PUF.csv,2026-04-04 15:50:22.777322+00:00
2,2014,AK,21989,HIOS,6,2014-03-19 07:06:49,21989,93-0438772,2014-01-01,2014-12-31,21989AK0020001,Rating Area 2,No Preference,Family Option,36.95,<NA>,73.90,107.61,107.61,107.61,144.56,144.56,144.56,15,s3://637423524537-eedb015-landing/raw/health_insurance/raw/2014/Rate_PUF.csv,2026-04-04 15:50:22.777322+00:00
3,2014,AK,21989,HIOS,6,2014-03-19 07:06:49,21989,93-0438772,2014-01-01,2014-12-31,21989AK0010001,Rating Area 1,No Preference,21,32.00,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,15,s3://637423524537-eedb015-landing/raw/health_insurance/raw/2014/Rate_PUF.csv,2026-04-04 15:50:22.777322+00:00
4,2014,AK,21989,HIOS,6,2014-03-19 07:06:49,21989,93-0438772,2014-01-01,2014-12-31,21989AK0010001,Rating Area 1,No Preference,22,32.00,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,16,s3://637423524537-eedb015-landing/raw/health_insurance/raw/2014/Rate_PUF.csv,2026-04-04 15:50:22.777322+00:00
5,2014,AK,21989,HIOS,6,2014-03-19 07:06:49,21989,93-0438772,2014-01-01,2014-12-31,21989AK0020001,Rating Area 3,No Preference,Family Option,36.95,<NA>,73.90,107.61,107.61,107.61,144.56,144.56,144.56,16,s3://637423524537-eedb015-landing/raw/health_insurance/raw/2014/Rate_PUF.csv,2026-04-04 15:50:22.777322+00:00
6,2014,AK,21989,HIOS,6,2014-03-19 07:06:49,21989,93-0438772,2014-01-01,2014-12-31,21989AK0020002,Rating Area 1,No Preference,Family Option,32.45,<NA>,64.90,94.50,94.50,94.50,126.95,126.95,126.95,17,s3://637423524537-eedb015-landing/raw/health_insurance/raw/2014/Rate_PUF.csv,2026-04-04 15:50:22.777322+00:00
7,2014,AK,21989,HIOS,6,2014-03-19 07:06:49,21989,93-0438772,2014-01-01,2014-12-31,21989AK0010001,Rating Area 1,No Preference,23,32.00,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,17,s3://637423524537-eedb015-landing/raw/health_insurance/raw/2014/Rate_PUF.csv,2026-04-04 15:50:22.777322+00:00
8,2014,AK,21989,HIOS,6,2014-03-19 07:06:49,21989,93-0438772,2014-01-01,2014-12-31,21989AK0010001,Rating Area 1,No Preference,24,32.00,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,18,s3://637423524537-eedb015-landing/raw/health_insurance/raw/2014/Rate_PUF.csv,2026-04-04 15:50:22.777322+00:00
9,2014,AK,21989,HIOS,6,2014-03-19 07:06:49,21989,93-0438772,2014-01-01,2014-12-31,21989AK0020002,Rating Area 2,No Preference,Family Option,32.45,<NA>,64.90,94.50,94.50,94.50,126.95,126.95,126.95,18,s3://637423524537-eedb015-landing/raw/health_insurance/raw/2014/Rate_PUF.csv,2026-04-04 15:50:22.777322+00:00


...

,businessyear,statecode,issuerid,sourcename,versionnum,importdate,issuerid2,federaltin,rateeffectivedate,rateexpirationdate,planid,ratingareaid,tobacco,age,individualrate,individualtobaccorate,couple,primarysubscriberandonedependent,primarysubscriberandtwodependents,primarysubscriberandthreeormoredependents,coupleandonedependent,coupleandtwodependents,coupleandthreeormoredependents,rownumber,landinzone_path,ingestion_datetime
0,2015,AK,21989,HIOS,4,2014-08-08 08:53:29,21989,93-0438772,2015-01-01,2015-12-31,21989AK0030001,Rating Area 1,No Preference,0-20,43.00,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,14,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Rate_PUF.csv,2026-04-04 15:52:44.304882+00:00
1,2015,AK,21989,HIOS,4,2014-08-08 08:53:29,21989,93-0438772,2015-01-01,2015-12-31,21989AK0030001,Rating Area 1,No Preference,21,38.00,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,15,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Rate_PUF.csv,2026-04-04 15:52:44.304882+00:00
2,2015,AK,21989,HIOS,4,2014-08-08 08:53:29,21989,93-0438772,2015-01-01,2015-12-31,21989AK0030001,Rating Area 1,No Preference,22,38.00,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,16,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Rate_PUF.csv,2026-04-04 15:52:44.304882+00:00
3,2015,AK,21989,HIOS,4,2014-08-08 08:53:29,21989,93-0438772,2015-01-01,2015-12-31,21989AK0030001,Rating Area 1,No Preference,23,38.00,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,17,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Rate_PUF.csv,2026-04-04 15:52:44.304882+00:00
4,2015,AK,21989,HIOS,4,2014-08-08 08:53:29,21989,93-0438772,2015-01-01,2015-12-31,21989AK0030001,Rating Area 1,No Preference,24,38.00,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,18,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Rate_PUF.csv,2026-04-04 15:52:44.304882+00:00
5,2015,AK,21989,HIOS,4,2014-08-08 08:53:29,21989,93-0438772,2015-01-01,2015-12-31,21989AK0030001,Rating Area 1,No Preference,25,38.00,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,19,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Rate_PUF.csv,2026-04-04 15:52:44.304882+00:00
6,2015,AK,21989,HIOS,4,2014-08-08 08:53:29,21989,93-0438772,2015-01-01,2015-12-31,21989AK0030001,Rating Area 1,No Preference,26,38.00,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,20,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Rate_PUF.csv,2026-04-04 15:52:44.304882+00:00
7,2015,AK,21989,HIOS,4,2014-08-08 08:53:29,21989,93-0438772,2015-01-01,2015-12-31,21989AK0030001,Rating Area 1,No Preference,27,38.00,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,21,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Rate_PUF.csv,2026-04-04 15:52:44.304882+00:00
8,2015,AK,21989,HIOS,4,2014-08-08 08:53:29,21989,93-0438772,2015-01-01,2015-12-31,21989AK0030001,Rating Area 1,No Preference,28,38.00,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,22,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Rate_PUF.csv,2026-04-04 15:52:44.304882+00:00
9,2015,AK,21989,HIOS,4,2014-08-08 08:53:29,21989,93-0438772,2015-01-01,2015-12-31,21989AK0030001,Rating Area 1,No Preference,29,38.00,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,23,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Rate_PUF.csv,2026-04-04 15:52:44.304882+00:00


   ↳ 26 colunas | 10 linhas exibidas



### `raw_rate_puf_2015_12_08`

,businessyear,statecode,issuerid,sourcename,versionnum,importdate,issuerid2,federaltin,rateeffectivedate,rateexpirationdate,planid,ratingareaid,tobacco,age,individualrate,individualtobaccorate,couple,primarysubscriberandonedependent,primarysubscriberandtwodependents,primarysubscriberandthreeormoredependents,coupleandonedependent,coupleandtwodependents,coupleandthreeormoredependents,rownumber,landinzone_path,ingestion_datetime
0,2016,AK,21989,HIOS,4,2015-08-22 15:09:32,21989,93-0438772,2016-01-01,2016-12-31,21989AK0030001,Rating Area 1,No Preference,0-20,43.00,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,14,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Rate_PUF_2015-12-08.csv,2026-04-04 15:53:04.369102+00:00
1,2016,AK,21989,HIOS,4,2015-08-22 15:09:32,21989,93-0438772,2016-01-01,2016-03-31,21989AK0090001,Rating Area 1,No Preference,0-20,61.00,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,14,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Rate_PUF_2015-12-08.csv,2026-04-04 15:53:04.369102+00:00
2,2016,AK,21989,HIOS,4,2015-08-22 15:09:32,21989,93-0438772,2016-04-01,2016-06-30,21989AK0090001,Rating Area 1,No Preference,0-20,62.00,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,14,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Rate_PUF_2015-12-08.csv,2026-04-04 15:53:04.369102+00:00
3,2016,AK,21989,HIOS,4,2015-08-22 15:09:32,21989,93-0438772,2016-10-01,2016-12-31,21989AK0090001,Rating Area 1,No Preference,0-20,64.00,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,14,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Rate_PUF_2015-12-08.csv,2026-04-04 15:53:04.369102+00:00
4,2016,AK,21989,HIOS,4,2015-08-22 15:09:32,21989,93-0438772,2016-07-01,2016-09-30,21989AK0090001,Rating Area 1,No Preference,0-20,63.00,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,14,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Rate_PUF_2015-12-08.csv,2026-04-04 15:53:04.369102+00:00
5,2016,AK,21989,HIOS,4,2015-08-22 15:09:32,21989,93-0438772,2016-01-01,2016-03-31,21989AK0080001,Rating Area 1,No Preference,Family Option,50.67,<NA>,100.33,116.04,116.04,116.04,117.86,117.86,117.86,14,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Rate_PUF_2015-12-08.csv,2026-04-04 15:53:04.369102+00:00
6,2016,AK,21989,HIOS,4,2015-08-22 15:09:32,21989,93-0438772,2016-04-01,2016-06-30,21989AK0080001,Rating Area 1,No Preference,Family Option,51.36,<NA>,101.70,117.62,117.62,117.62,180.28,180.28,180.28,14,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Rate_PUF_2015-12-08.csv,2026-04-04 15:53:04.369102+00:00
7,2016,AK,21989,HIOS,4,2015-08-22 15:09:32,21989,93-0438772,2016-07-01,2016-09-30,21989AK0080001,Rating Area 1,No Preference,Family Option,52.06,<NA>,103.08,119.22,119.22,119.22,182.73,182.73,182.73,14,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Rate_PUF_2015-12-08.csv,2026-04-04 15:53:04.369102+00:00
8,2016,AK,21989,HIOS,4,2015-08-22 15:09:32,21989,93-0438772,2016-10-01,2016-12-31,21989AK0080001,Rating Area 1,No Preference,Family Option,52.77,<NA>,104.48,120.84,120.84,120.84,185.22,185.22,185.22,14,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Rate_PUF_2015-12-08.csv,2026-04-04 15:53:04.369102+00:00
9,2016,AK,21989,HIOS,4,2015-08-22 15:09:32,21989,93-0438772,2016-10-01,2016-12-31,21989AK0080001,Rating Area 2,No Preference,Family Option,52.77,<NA>,104.48,120.84,120.84,120.84,185.22,185.22,185.22,15,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Rate_PUF_2015-12-08.csv,2026-04-04 15:53:04.369102+00:00


   ↳ 26 colunas | 10 linhas exibidas



---
## 4. Tabela: PlanAttributes — Atributos dos Planos

**O que é:** Metadados descritivos de cada plano: nível metálico, tipo de rede, deductibles, out-of-pocket máximos, cobertura de saúde mental, dental, visão, etc. Atua como o **catálogo mestre dos planos** — a tabela dimensional central do dataset (~11.000 planos por ano).

**Colunas principais:**

| Coluna | Tipo | Descrição |
|---|---|---|
| `BusinessYear` | integer | Ano do plano |
| `StateCode` | string | Estado |
| `IssuerId` | string | Código da seguradora |
| `PlanId` | string | Identificador único HIOS do plano (chave primária) |
| `PlanMarketingName` | string | Nome comercial do plano (ex: "Blue Shield Silver 70") |
| `HIOSProductId` | string | ID do produto no sistema HIOS |
| `NetworkId` | string | Identificador da rede de prestadores |
| `ServiceAreaId` | string | ID da área de serviço — chave para `ServiceArea` |
| `FormularyId` | string | ID do formulário farmacêutico |
| `IsNewPlan` | string | Plano novo naquele ano (`Yes`/`No`) |
| `PlanType` | string | Tipo de rede: `HMO`, `PPO`, `EPO`, `POS`, `PFFS`, `HDHP` |
| `MetalLevel` | string | Nível metálico: `Bronze`, `Silver`, `Gold`, `Platinum`, `Catastrophic` |
| `QHPorSADPOnly` | string | Qualified Health Plan ou Standalone Dental Plan |
| `ChildOnlyOffering` | string | Plano disponível para crianças apenas |
| `WellnessProgramOffered` | string | Possui programa de bem-estar (`Yes`/`No`) |
| `DiseaseManagementProgramsOffered` | string | Programas de gestão de doenças crônicas |
| `NationalNetwork` | string | Cobertura nacional (`Yes`/`No`) |
| `EHBPercentTotalPremium` | decimal | % do prêmio referente a Essential Health Benefits |
| `CSRVariationType` | string | Tipo de variação de Cost-Sharing Reduction (para segurados de baixa renda) |
| `MEHBInnTier1IndividualMOOP` | decimal | Maximum Out-of-Pocket individual — médico in-network Tier 1 |
| `MEHBInnTier1FamilyMOOP` | decimal | MOOP familiar — médico in-network Tier 1 |
| `MEHBOutOfNetIndividualMOOP` | decimal | MOOP individual fora da rede |
| `MEHBInnTier1IndividualDeductible` | decimal | Deductible individual médico in-network |
| `MEHBInnTier1FamilyDeductible` | decimal | Deductible familiar médico in-network |
| `SBCHavingaBabyDeductible` | decimal | Exemplo SBC — custo de ter um bebê (deductible) |
| `SBCHavingDiabetesCopayment` | decimal | Exemplo SBC — copay anual para diabetes |
| `SBCHavingSimpleFractureDeductible` | decimal | Exemplo SBC — deductible para fratura simples |

**Nota sobre `PlanId` (formato HIOS):** `XXXXX-XX-XXXXXXXX` — `IssuerId (5 dígitos) + ProductId (2) + PlanId (2) + VariantId (2)`. Variantes com sufixo `73` representam o mesmo plano com deductibles reduzidos para elegíveis ao subsídio CSR — devem ser agrupadas com o plano base nos joins.

**Relações:** Tabela dimensional central. Liga-se com `Rate` (via `PlanId`), `BenefitsCostSharing` (via `PlanId`), `ServiceArea` (via `ServiceAreaId`) e `Network` (via `NetworkId`).

**Insights dos notebooks Kaggle:**
- `MetalLevel = Silver` é o mais comum (~40% dos planos) por ser o nível de referência para os subsídios federais (Cost-Sharing Reductions).
- HMOs dominam mercados urbanos densos; PPOs predominam em estados com menor densidade de prestadores.
- O número de `IssuerId` distintos por `StateCode` é o principal proxy de competição para a Questão 2.

**Relevância para o projeto:** Chave de segmentação em todas as questões — `MetalLevel` e `PlanType` são as variáveis de classificação mais usadas nas análises.

In [6]:
show_family(["plan_attributes"])

### `plan_attributes`

,avcalculatoroutputnumber,beginprimarycarecostsharingafternumberofvisits,beginprimarycaredeductiblecoinsuranceafternumberofcopays,benefitpackageid,businessyear,csrvariationtype,childonlyoffering,childonlyplanid,compositeratingoffered,dehbcombinnoonfamilymoop,dehbcombinnoonfamilypergroupmoop,dehbcombinnoonfamilyperpersonmoop,dehbcombinnoonindividualmoop,dehbdedcombinnoonfamily,dehbdedcombinnoonfamilypergroup,dehbdedcombinnoonfamilyperperson,dehbdedcombinnoonindividual,dehbdedinntier1coinsurance,dehbdedinntier1family,dehbdedinntier1familypergroup,dehbdedinntier1familyperperson,dehbdedinntier1individual,dehbdedinntier2coinsurance,dehbdedinntier2family,dehbdedinntier2familypergroup,dehbdedinntier2familyperperson,dehbdedinntier2individual,dehbdedoutofnetfamily,dehbdedoutofnetfamilypergroup,dehbdedoutofnetfamilyperperson,dehbdedoutofnetindividual,dehbinntier1familymoop,dehbinntier1familypergroupmoop,dehbinntier1familyperpersonmoop,dehbinntier1individualmoop,dehbinntier2familymoop,dehbinntier2familypergroupmoop,dehbinntier2familyperpersonmoop,dehbinntier2individualmoop,dehboutofnetfamilymoop,dehboutofnetfamilypergroupmoop,dehboutofnetfamilyperpersonmoop,dehboutofnetindividualmoop,dentalonlyplan,diseasemanagementprogramsoffered,ehbpediatricdentalapportionmentquantity,ehbpercentpremiums4,ehbpercenttotalpremium,firsttierutilization,formularyid,formularyurl,hiosproductid,hpid,hsaorhraemployercontribution,hsaorhraemployercontributionamount,importdate,indianplanvariationestimatedadvancedpaymentamountperenrollee,inpatientcopaymentmaximumdays,isguaranteedrate,ishsaeligible,isnewplan,isnoticerequiredforpregnancy,isreferralrequiredforspecialist,issueractuarialvalue,issuerid,issuerid2,mehbcombinnoonfamilymoop,mehbcombinnoonfamilypergroupmoop,mehbcombinnoonfamilyperpersonmoop,mehbcombinnoonindividualmoop,mehbdedcombinnoonfamily,mehbdedcombinnoonfamilypergroup,mehbdedcombinnoonfamilyperperson,mehbdedcombinnoonindividual,mehbdedinntier1coinsurance,mehbdedinntier1family,mehbdedinntier1familypergroup,mehbdedinntier1familyperperson,mehbdedinntier1individual,mehbdedinntier2coinsurance,mehbdedinntier2family,mehbdedinntier2familypergroup,mehbdedinntier2familyperperson,mehbdedinntier2individual,mehbdedoutofnetfamily,mehbdedoutofnetfamilypergroup,mehbdedoutofnetfamilyperperson,mehbdedoutofnetindividual,mehbinntier1familymoop,mehbinntier1familypergroupmoop,mehbinntier1familyperpersonmoop,mehbinntier1individualmoop,mehbinntier2familymoop,mehbinntier2familypergroupmoop,mehbinntier2familyperpersonmoop,mehbinntier2individualmoop,mehboutofnetfamilymoop,mehboutofnetfamilypergroupmoop,mehboutofnetfamilyperpersonmoop,mehboutofnetindividualmoop,marketcoverage,medicaldrugdeductiblesintegrated,medicaldrugmaximumoutofpocketintegrated,metallevel,multipleinnetworktiers,nationalnetwork,networkid,outofcountrycoverage,outofcountrycoveragedescription,outofserviceareacoverage,outofserviceareacoveragedescription,planbrochure,planeffictivedate,planexpirationdate,planid,planlevelexclusions,planmarketingname,plantype,qhpnonqhptypeid,rownumber,sbchavingdiabetescoinsurance,sbchavingdiabetescopayment,sbchavingdiabetesdeductible,sbchavingdiabeteslimit,sbchavingababycoinsurance,sbchavingababycopayment,sbchavingababydeductible,sbchavingababylimit,secondtierutilization,serviceareaid,sourcename,specialistrequiringreferral,specialtydrugmaximumcoinsurance,standardcomponentid,statecode,statecode2,tehbcombinnoonfamilymoop,tehbcombinnoonfamilypergroupmoop,tehbcombinnoonfamilyperpersonmoop,tehbcombinnoonindividualmoop,tehbdedcombinnoonfamily,tehbdedcombinnoonfamilypergroup,tehbdedcombinnoonfamilyperperson,tehbdedcombinnoonindividual,tehbdedinntier1coinsurance,tehbdedinntier1family,tehbdedinntier1familypergroup,tehbdedinntier1familyperperson,tehbdedinntier1individual,tehbdedinntier2coinsurance,tehbdedinntier2family,tehbdedinntier2familypergroup,tehbdedinntier2familyperperson,tehbdedinntier2individual,tehbdedoutofnetfamily,tehbdedoutofnetfamilypergroup,tehbdedoutofnetfamilyperperson,tehbdedouto

   ↳ 178 colunas | 10 linhas exibidas



### `raw_2014_plan_attributes_puf_2014_2015_03_09`

,businessyear,statecode,issuerid,sourcename,versionnum,importdate,benefitpackageid,issuerid2,statecode2,marketcoverage,dentalonlyplan,tin,standardcomponentid,planmarketingname,hiosproductid,hpid,networkid,serviceareaid,formularyid,isnewplan,plantype,metallevel,uniqueplandesign,qhpnonqhptypeid,isnoticerequiredforpregnancy,isreferralrequiredforspecialist,specialistrequiringreferral,planlevelexclusions,indianplanvariationestimatedadvancedpaymentamountperenrollee,ishsaeligible,hsaorhraemployercontribution,hsaorhraemployercontributionamount,childonlyoffering,childonlyplanid,wellnessprogramoffered,diseasemanagementprogramsoffered,ehbpediatricdentalapportionmentquantity,ehbpercentpremiums4,isguaranteedrate,specialtydrugmaximumcoinsurance,inpatientcopaymentmaximumdays,beginprimarycarecostsharingafternumberofvisits,beginprimarycaredeductiblecoinsuranceafternumberofcopays,planeffictivedate,planexpirationdate,outofcountrycoverage,outofcountrycoveragedescription,outofserviceareacoverage,outofserviceareacoveragedescription,nationalnetwork,urlforsummaryofbenefitscoverage,urlforenrollmentpayment,planbrochure,formularyurl,planid,csrvariationtype,issueractuarialvalue,avcalculatoroutputnumber,medicaldrugdeductiblesintegrated,medicaldrugmaximumoutofpocketintegrated,multipleinnetworktiers,firsttierutilization,secondtierutilization,mehbinntier1individualmoop,mehbinntier1familymoop,mehbinntier2individualmoop,mehbinntier2familymoop,mehboutofnetindividualmoop,mehboutofnetfamilymoop,mehbcombinnoonindividualmoop,mehbcombinnoonfamilymoop,dehbinntier1individualmoop,dehbinntier1familymoop,dehbinntier2individualmoop,dehbinntier2familymoop,dehboutofnetindividualmoop,dehboutofnetfamilymoop,dehbcombinnoonindividualmoop,dehbcombinnoonfamilymoop,tehbinntier1individualmoop,tehbinntier1familymoop,tehbinntier2individualmoop,tehbinntier2familymoop,tehboutofnetindividualmoop,tehboutofnetfamilymoop,tehbcombinnoonindividualmoop,tehbcombinnoonfamilymoop,mehbdedinntier1individual,mehbdedinntier1family,mehbdedinntier1coinsurance,mehbdedinntier2individual,mehbdedinntier2family,mehbdedinntier2coinsurance,mehbdedoutofnetindividual,mehbdedoutofnetfamily,mehbdedcombinnoonindividual,mehbdedcombinnoonfamily,dehbdedinntier1individual,dehbdedinntier1family,dehbdedinntier1coinsurance,dehbdedinntier2individual,dehbdedinntier2family,dehbdedinntier2coinsurance,dehbdedoutofnetindividual,dehbdedoutofnetfamily,dehbdedcombinnoonindividual,dehbdedcombinnoonfamily,tehbdedinntier1individual,tehbdedinntier1family,tehbdedinntier1coinsurance,tehbdedinntier2individual,tehbdedinntier2family,tehbdedinntier2coinsurance,tehbdedoutofnetindividual,tehbdedoutofnetfamily,tehbdedcombinnoonindividual,tehbdedcombinnoonfamily,sbchavingababydeductible,sbchavingababycopayment,sbchavingababycoinsurance,sbchavingababylimit,sbchavingdiabetesdeductible,sbchavingdiabetescopayment,sbchavingdiabetescoinsurance,sbchavingdiabeteslimit,rownumber,landinzone_path,ingestion_datetime
0,2014,AK,21989,HIOS,6,2014-03-19 07:06:49,1,21989,AK,SHOP (Small Group),Yes,93-0438772,21989AK0020002,Premier,21989AK002,<NA>,AKN001,AKS001,<NA>,New,PPO,Low,<NA>,Both,<NA>,<NA>,<NA>,OOP Max only applies to pediatric benefits,<NA>,<NA>,<NA>,<NA>,Allows Adult and Child-Only,<NA>,<NA>,<NA>,29,<NA>,Guaranteed Rate,<NA>,0,0,0,2014-01-01,2014-12-31,No,<NA>,Yes,National Network,Yes,https://www.modahealth.com/producers/grp/den.shtml,https://www.modahealth.com/employers/enroll.shtml,https://www.modahealth.com/producers/grp/den.shtml,<NA>,21989AK0020002-00,Standard Low Off Exchange Plan,70.00%,<NA>,<NA>,<NA>,No,100%,<NA>,Not Applicable,Not Applicable,<NA>,<NA>,Not Applicable,Not Applicable,$700,"$1,400",<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,Not Applicable,Not Applicable,<NA>,<NA>,<NA>,<NA>,Not Applicable,Not Applicable,$50,$100,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,6,s3://637423524537-eedb015-landing/raw/healt

...

,businessyear,statecode,issuerid,sourcename,versionnum,importdate,benefitpackageid,issuerid2,statecode2,marketcoverage,dentalonlyplan,tin,standardcomponentid,planmarketingname,hiosproductid,hpid,networkid,serviceareaid,formularyid,isnewplan,plantype,metallevel,uniqueplandesign,qhpnonqhptypeid,isnoticerequiredforpregnancy,isreferralrequiredforspecialist,specialistrequiringreferral,planlevelexclusions,indianplanvariationestimatedadvancedpaymentamountperenrollee,ishsaeligible,hsaorhraemployercontribution,hsaorhraemployercontributionamount,childonlyoffering,childonlyplanid,wellnessprogramoffered,diseasemanagementprogramsoffered,ehbpediatricdentalapportionmentquantity,ehbpercentpremiums4,isguaranteedrate,specialtydrugmaximumcoinsurance,inpatientcopaymentmaximumdays,beginprimarycarecostsharingafternumberofvisits,beginprimarycaredeductiblecoinsuranceafternumberofcopays,planeffictivedate,planexpirationdate,outofcountrycoverage,outofcountrycoveragedescription,outofserviceareacoverage,outofserviceareacoveragedescription,nationalnetwork,urlforsummaryofbenefitscoverage,urlforenrollmentpayment,planbrochure,formularyurl,planid,csrvariationtype,issueractuarialvalue,avcalculatoroutputnumber,medicaldrugdeductiblesintegrated,medicaldrugmaximumoutofpocketintegrated,multipleinnetworktiers,firsttierutilization,secondtierutilization,mehbinntier1individualmoop,mehbinntier1familymoop,mehbinntier2individualmoop,mehbinntier2familymoop,mehboutofnetindividualmoop,mehboutofnetfamilymoop,mehbcombinnoonindividualmoop,mehbcombinnoonfamilymoop,dehbinntier1individualmoop,dehbinntier1familymoop,dehbinntier2individualmoop,dehbinntier2familymoop,dehboutofnetindividualmoop,dehboutofnetfamilymoop,dehbcombinnoonindividualmoop,dehbcombinnoonfamilymoop,tehbinntier1individualmoop,tehbinntier1familymoop,tehbinntier2individualmoop,tehbinntier2familymoop,tehboutofnetindividualmoop,tehboutofnetfamilymoop,tehbcombinnoonindividualmoop,tehbcombinnoonfamilymoop,mehbdedinntier1individual,mehbdedinntier1family,mehbdedinntier1coinsurance,mehbdedinntier2individual,mehbdedinntier2family,mehbdedinntier2coinsurance,mehbdedoutofnetindividual,mehbdedoutofnetfamily,mehbdedcombinnoonindividual,mehbdedcombinnoonfamily,dehbdedinntier1individual,dehbdedinntier1family,dehbdedinntier1coinsurance,dehbdedinntier2individual,dehbdedinntier2family,dehbdedinntier2coinsurance,dehbdedoutofnetindividual,dehbdedoutofnetfamily,dehbdedcombinnoonindividual,dehbdedcombinnoonfamily,tehbdedinntier1individual,tehbdedinntier1family,tehbdedinntier1coinsurance,tehbdedinntier2individual,tehbdedinntier2family,tehbdedinntier2coinsurance,tehbdedoutofnetindividual,tehbdedoutofnetfamily,tehbdedcombinnoonindividual,tehbdedcombinnoonfamily,sbchavingababydeductible,sbchavingababycopayment,sbchavingababycoinsurance,sbchavingababylimit,sbchavingdiabetesdeductible,sbchavingdiabetescopayment,sbchavingdiabetescoinsurance,sbchavingdiabeteslimit,rownumber,landinzone_path,ingestion_datetime
0,2014,AK,21989,HIOS,6,2014-03-19 07:06:49,1,21989,AK,SHOP (Small Group),Yes,93-0438772,21989AK0020002,Premier,21989AK002,<NA>,AKN001,AKS001,<NA>,New,PPO,Low,<NA>,Both,<NA>,<NA>,<NA>,OOP Max only applies to pediatric benefits,<NA>,<NA>,<NA>,<NA>,Allows Adult and Child-Only,<NA>,<NA>,<NA>,29,<NA>,Guaranteed Rate,<NA>,0,0,0,2014-01-01,2014-12-31,No,<NA>,Yes,National Network,Yes,https://www.modahealth.com/producers/grp/den.shtml,https://www.modahealth.com/employers/enroll.shtml,https://www.modahealth.com/producers/grp/den.shtml,<NA>,21989AK0020002-00,Standard Low Off Exchange Plan,70.00%,<NA>,<NA>,<NA>,No,100%,<NA>,Not Applicable,Not Applicable,<NA>,<NA>,Not Applicable,Not Applicable,$700,"$1,400",<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,Not Applicable,Not Applicable,<NA>,<NA>,<NA>,<NA>,Not Applicable,Not Applicable,$50,$100,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,6,s3://637423524537-eedb015-landing/raw/healt

   ↳ 128 colunas | 10 linhas exibidas



### `raw_plan_attributes_puf_2015_12_08`

,businessyear,statecode,issuerid,sourcename,versionnum,importdate,benefitpackageid,issuerid2,statecode2,marketcoverage,dentalonlyplan,tin,standardcomponentid,planmarketingname,hiosproductid,hpid,networkid,serviceareaid,formularyid,isnewplan,plantype,metallevel,uniqueplandesign,qhpnonqhptypeid,isnoticerequiredforpregnancy,isreferralrequiredforspecialist,specialistrequiringreferral,planlevelexclusions,indianplanvariationestimatedadvancedpaymentamountperenrollee,compositeratingoffered,childonlyoffering,childonlyplanid,wellnessprogramoffered,diseasemanagementprogramsoffered,ehbpercenttotalpremium,ehbpediatricdentalapportionmentquantity,isguaranteedrate,specialtydrugmaximumcoinsurance,inpatientcopaymentmaximumdays,beginprimarycarecostsharingafternumberofvisits,beginprimarycaredeductiblecoinsuranceafternumberofcopays,planeffictivedate,planexpirationdate,outofcountrycoverage,outofcountrycoveragedescription,outofserviceareacoverage,outofserviceareacoveragedescription,nationalnetwork,urlforenrollmentpayment,formularyurl,planid,csrvariationtype,issueractuarialvalue,avcalculatoroutputnumber,medicaldrugdeductiblesintegrated,medicaldrugmaximumoutofpocketintegrated,multipleinnetworktiers,firsttierutilization,secondtierutilization,sbchavingababydeductible,sbchavingababycopayment,sbchavingababycoinsurance,sbchavingababylimit,sbchavingdiabetesdeductible,sbchavingdiabetescopayment,sbchavingdiabetescoinsurance,sbchavingdiabeteslimit,mehbinntier1individualmoop,mehbinntier1familyperpersonmoop,mehbinntier1familypergroupmoop,mehbinntier2individualmoop,mehbinntier2familyperpersonmoop,mehbinntier2familypergroupmoop,mehboutofnetindividualmoop,mehboutofnetfamilyperpersonmoop,mehboutofnetfamilypergroupmoop,mehbcombinnoonindividualmoop,mehbcombinnoonfamilyperpersonmoop,mehbcombinnoonfamilypergroupmoop,dehbinntier1individualmoop,dehbinntier1familyperpersonmoop,dehbinntier1familypergroupmoop,dehbinntier2individualmoop,dehbinntier2familyperpersonmoop,dehbinntier2familypergroupmoop,dehboutofnetindividualmoop,dehboutofnetfamilyperpersonmoop,dehboutofnetfamilypergroupmoop,dehbcombinnoonindividualmoop,dehbcombinnoonfamilyperpersonmoop,dehbcombinnoonfamilypergroupmoop,tehbinntier1individualmoop,tehbinntier1familyperpersonmoop,tehbinntier1familypergroupmoop,tehbinntier2individualmoop,tehbinntier2familyperpersonmoop,tehbinntier2familypergroupmoop,tehboutofnetindividualmoop,tehboutofnetfamilyperpersonmoop,tehboutofnetfamilypergroupmoop,tehbcombinnoonindividualmoop,tehbcombinnoonfamilyperpersonmoop,tehbcombinnoonfamilypergroupmoop,mehbdedinntier1individual,mehbdedinntier1familyperperson,mehbdedinntier1familypergroup,mehbdedinntier1coinsurance,mehbdedinntier2individual,mehbdedinntier2familyperperson,mehbdedinntier2familypergroup,mehbdedinntier2coinsurance,mehbdedoutofnetindividual,mehbdedoutofnetfamilyperperson,mehbdedoutofnetfamilypergroup,mehbdedcombinnoonindividual,mehbdedcombinnoonfamilyperperson,mehbdedcombinnoonfamilypergroup,dehbdedinntier1individual,dehbdedinntier1familyperperson,dehbdedinntier1familypergroup,dehbdedinntier1coinsurance,dehbdedinntier2individual,dehbdedinntier2familyperperson,dehbdedinntier2familypergroup,dehbdedinntier2coinsurance,dehbdedoutofnetindividual,dehbdedoutofnetfamilyperperson,dehbdedoutofnetfamilypergroup,dehbdedcombinnoonindividual,dehbdedcombinnoonfamilyperperson,dehbdedcombinnoonfamilypergroup,tehbdedinntier1individual,tehbdedinntier1familyperperson,tehbdedinntier1familypergroup,tehbdedinntier1coinsurance,tehbdedinntier2individual,tehbdedinntier2familyperperson,tehbdedinntier2familypergroup,tehbdedinntier2coinsurance,tehbdedoutofnetindividual,tehbdedoutofnetfamilyperperson,tehbdedoutofnetfamilypergroup,tehbdedcombinnoonindividual,tehbdedcombinnoonfamilyperperson,tehbdedcombinnoonfamilypergroup,ishsaeligible,hsaorhraemployercontribution,hsaorhraemployercontributionamount,urlforsummaryofbenefitscoverage,planbrochure,rownumber,landinzone_path,ingestion_datetime
0,2016,AK,21989,HIOS,4,8/22/2015 15:09,1,21989,AK,Individual,Yes,93-0438

   ↳ 153 colunas | 10 linhas exibidas



---
## 5. Tabela: BenefitsCostSharing — Benefícios e Compartilhamento de Custos

**O que é:** Detalha os **benefícios cobertos** por cada plano e as regras de compartilhamento de custo que o segurado paga ao utilizar cada serviço. Cada linha representa um benefício específico dentro de um plano — é a tabela mais granular do dataset em termos de coberturas.

**Colunas principais:**

| Coluna | Tipo | Descrição |
|---|---|---|
| `BusinessYear` | integer | Ano do plano |
| `StateCode` | string | Estado |
| `IssuerId` | string | ID da seguradora |
| `PlanId` | string | Chave de ligação com `PlanAttributes` |
| `BenefitName` | string | Nome do benefício (ver lista abaixo) |
| `IsCovered` | string | Se o benefício é coberto: `Covered` / `Not Covered` |
| `QuantLimitOnSvc` | string | Há limite quantitativo de serviços? (`Yes`/`No`) |
| `LimitQty` | decimal | Quantidade máxima coberta (ex: 30 visitas/ano) |
| `LimitUnit` | string | Unidade do limite: `Per Year`, `Per Visit`, `Days` |
| `Exclusions` | string | Descrição de exclusões específicas |
| `Explanation` | string | Explicação adicional sobre o benefício |
| `EHBInd` | string | É um Essential Health Benefit obrigatório pelo ACA? |
| `IsSubjToDedBefore` | string | O cost-sharing está sujeito ao deductible antes de ser aplicado? |
| `CoinsInnTier1` | string | Coinsurance (%) — prestador in-network Tier 1 |
| `CoinsInnTier2` | string | Coinsurance (%) — prestador in-network Tier 2 |
| `CoinsOutofNet` | string | Coinsurance (%) — prestador fora da rede |
| `CopayInnTier1` | string | Copay ($) — prestador in-network Tier 1 |
| `CopayInnTier2` | string | Copay ($) — prestador in-network Tier 2 |
| `CopayOutofNet` | string | Copay ($) — prestador fora da rede |

**Benefícios de maior relevância para o projeto:**

| BenefitName | Relevância |
|---|---|
| `Chemotherapy` | Questão 1 — evolução do custo para pacientes oncológicos |
| `Radiation Therapy` | Questão 1 — custo de radioterapia ao longo dos anos |
| `Infusion Therapy` | Questão 1 — terapia recorrente para doenças crônicas |
| `Specialist Visit` | Questão 3 — benefício como variável de precificação |
| `Primary Care Visit to Treat an Injury or Illness` | Questão 3 |
| `Emergency Room Services` | Questão 3 |
| `Mental/Behavioral Health Outpatient Services` | Questão 3 |

**Relações:** `PlanId` → `PlanAttributes.PlanId` (e transitivamente → `Rate.PlanId`).

**Insights dos notebooks Kaggle (Ryan Burge — Cancer and the ACA):**
- Entre 2014 e 2016, a proporção de planos com **coinsurance zero** para quimioterapia aumentou — melhora geral na cobertura oncológica.
- Planos **Gold** e **Platinum** tendem a ter copay fixo para quimioterapia, protegendo o paciente de custos variáveis em tratamentos longos.
- Planos **Bronze** frequentemente aplicam coinsurance de 40–50% após o deductible, expondo pacientes crônicos a custos catastróficos.
- `CopayInnTier2` e `CoinsInnTier2` têm alta taxa de nulos — muitos planos usam apenas Tier 1.

**Atenção ao parsing:** Os campos de copay/coinsurance chegam como strings mistas: `"$30"`, `"$0"`, `"No Charge"`, `"Not Applicable"`. O ETL da camada Silver precisa normalizar esses valores para decimais.

**Relevância para o projeto:** **Principal tabela para a Questão 1** (evolução Copay × Coinsurance em oncologia) e fundamental para a Questão 3 (benefícios como variável de precificação).

In [7]:
show_family(["benefits_cost_sharing", "benefits"])

### `benefits_cost_sharing`

,benefitname,businessyear,coinsinntier1,coinsinntier2,coinsoutofnet,copayinntier1,copayinntier2,copayoutofnet,ehbvarreason,exclusions,explanation,importdate,iscovered,isehb,isexclfrominnmoop,isexclfromoonmoop,isstatemandate,issubjtodedtier1,issubjtodedtier2,issuerid,issuerid2,limitqty,limitunit,minimumstay,planid,quantlimitonsvc,rownumber,sourcename,standardcomponentid,statecode,statecode2,versionnum,landinzone_path,ingestion_datetime
0,Routine Dental Services (Adult),2014,20%,<NA>,20%,No Charge,<NA>,No Charge,Above EHB,<NA>,Combined annual benefit maximum of $1000 per year for adults. See policy for additional limitations,2014-03-19 07:06:49,Covered,<NA>,No,No,<NA>,No,No,21989,21989,1000.0,Dollars per Year,<NA>,21989AK0010001-00,Yes,68,HIOS,21989AK0010001,AK,AK,6,s3://637423524537-eedb015-landing/raw/health_insurance/BenefitsCostSharing.csv,2026-04-04 15:48:30.673734+00:00
1,Dental Check-Up for Children,2014,20%,<NA>,20%,No Charge,<NA>,No Charge,Substantially Equal,<NA>,See policy for additional limitations,2014-03-19 07:06:49,Covered,Yes,No,No,<NA>,No,No,21989,21989,1.0,Visit(s) per 6 Months,<NA>,21989AK0010001-00,Yes,104,HIOS,21989AK0010001,AK,AK,6,s3://637423524537-eedb015-landing/raw/health_insurance/BenefitsCostSharing.csv,2026-04-04 15:48:30.673734+00:00
2,Basic Dental Care - Child,2014,40%,<NA>,40%,No Charge,<NA>,No Charge,Substantially Equal,<NA>,See policy for additional limitations,2014-03-19 07:06:49,Covered,Yes,No,No,<NA>,Yes,Yes,21989,21989,<NA>,<NA>,<NA>,21989AK0010001-00,<NA>,110,HIOS,21989AK0010001,AK,AK,6,s3://637423524537-eedb015-landing/raw/health_insurance/BenefitsCostSharing.csv,2026-04-04 15:48:30.673734+00:00
3,Orthodontia - Child,2014,50%,<NA>,50%,No Charge,<NA>,No Charge,Additional EHB Benefit,<NA>,"24 month waiting period, See policy for additional limitations",2014-03-19 07:06:49,Covered,<NA>,No,No,<NA>,Yes,Yes,21989,21989,<NA>,<NA>,<NA>,21989AK0010001-00,<NA>,111,HIOS,21989AK0010001,AK,AK,6,s3://637423524537-eedb015-landing/raw/health_insurance/BenefitsCostSharing.csv,2026-04-04 15:48:30.673734+00:00
4,Major Dental Care - Child,2014,50%,<NA>,50%,No Charge,<NA>,No Charge,Substantially Equal,<NA>,See policy for additional limitations,2014-03-19 07:06:49,Covered,Yes,No,No,<NA>,Yes,Yes,21989,21989,<NA>,<NA>,<NA>,21989AK0010001-00,<NA>,112,HIOS,21989AK0010001,AK,AK,6,s3://637423524537-eedb015-landing/raw/health_insurance/BenefitsCostSharing.csv,2026-04-04 15:48:30.673734+00:00
5,Basic Dental Care - Adult,2014,40%,<NA>,40%,No Charge,<NA>,No Charge,Above EHB,<NA>,"Combined annual benefit maximum of $1000 per year for adults, 6 month waiting period, See policy...",2014-03-19 07:06:49,Covered,<NA>,No,No,<NA>,Yes,Yes,21989,21989,1000.0,Dollars per Year,<NA>,21989AK0010001-00,Yes,113,HIOS,21989AK0010001,AK,AK,6,s3://637423524537-eedb015-landing/raw/health_insurance/BenefitsCostSharing.csv,2026-04-04 15:48:30.673734+00:00
6,Orthodontia - Adult,2014,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2014-03-19 07:06:49,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,21989,21989,<NA>,<NA>,<NA>,21989AK0010001-00,<NA>,114,HIOS,21989AK0010001,AK,AK,6,s3://637423524537-eedb015-landing/raw/health_insurance/BenefitsCostSharing.csv,2026-04-04 15:48:30.673734+00:00
7,Major Dental Care - Adult,2014,50%,<NA>,50%,No Charge,<NA>,No Charge,Above EHB,<NA>,"Combined annual benefit maximum of $1000 per year for adults, 12 month waiting period, See polic...",2014-03-19 07:06:49,Covered,<NA>,No,No,<NA>,Yes,Yes,21989,21989,1000.0,Dollars per Year,<NA>,21989AK0010001-00,Yes,115,HIOS,21989AK0010001,AK,AK,6,s3://637423524537-eedb015-landing/raw/health_insurance/BenefitsCostSharing.csv,2026-04-04 15:48:30.673734+00:00
8,Accidental Dental,2014,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2014-03-19 07:06:49,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,21989,21989,<NA>,<NA>,<NA>,21989AK0010001-00,<NA>,118,HIOS,21989AK0010001,AK,AK,6,s3://637423524537-eedb015-landing/raw/health_insurance/BenefitsCostSharing.csv,2026-04-04 15:48:30.673734+00:00
9,Routine Dental Servi

   ↳ 34 colunas | 10 linhas exibidas



### `raw_2014_benefits_cost_sharing_puf`

,businessyear,statecode,issuerid,sourcename,versionnum,importdate,issuerid2,statecode2,standardcomponentid,planid,benefitname,copayinntier1,copayinntier2,copayoutofnet,coinsinntier1,coinsinntier2,coinsoutofnet,isehb,isstatemandate,iscovered,quantlimitonsvc,limitqty,limitunit,minimumstay,exclusions,explanation,ehbvarreason,issubjtodedtier1,issubjtodedtier2,isexclfrominnmoop,isexclfromoonmoop,rownumber,landinzone_path,ingestion_datetime
0,2014,AK,21989,HIOS,6,2014-03-19 07:06:49,21989,AK,21989AK0010001,21989AK0010001-00,Routine Dental Services (Adult),No Charge,<NA>,No Charge,20%,<NA>,20%,<NA>,<NA>,Covered,Yes,1000,Dollars per Year,<NA>,<NA>,Combined annual benefit maximum of $1000 per year for adults. See policy for additional limitations,Above EHB,No,No,No,No,68,s3://637423524537-eedb015-landing/raw/health_insurance/raw/2014/Benefits_Cost_Sharing_PUF.csv,2026-04-04 15:50:07.877589+00:00
1,2014,AK,21989,HIOS,6,2014-03-19 07:06:49,21989,AK,21989AK0010001,21989AK0010001-00,Dental Check-Up for Children,No Charge,<NA>,No Charge,20%,<NA>,20%,Yes,<NA>,Covered,Yes,1,Visit(s) per 6 Months,<NA>,<NA>,See policy for additional limitations,Substantially Equal,No,No,No,No,104,s3://637423524537-eedb015-landing/raw/health_insurance/raw/2014/Benefits_Cost_Sharing_PUF.csv,2026-04-04 15:50:07.877589+00:00
2,2014,AK,21989,HIOS,6,2014-03-19 07:06:49,21989,AK,21989AK0010001,21989AK0010001-00,Basic Dental Care - Child,No Charge,<NA>,No Charge,40%,<NA>,40%,Yes,<NA>,Covered,<NA>,<NA>,<NA>,<NA>,<NA>,See policy for additional limitations,Substantially Equal,Yes,Yes,No,No,110,s3://637423524537-eedb015-landing/raw/health_insurance/raw/2014/Benefits_Cost_Sharing_PUF.csv,2026-04-04 15:50:07.877589+00:00
3,2014,AK,21989,HIOS,6,2014-03-19 07:06:49,21989,AK,21989AK0010001,21989AK0010001-00,Orthodontia - Child,No Charge,<NA>,No Charge,50%,<NA>,50%,<NA>,<NA>,Covered,<NA>,<NA>,<NA>,<NA>,<NA>,"24 month waiting period, See policy for additional limitations",Additional EHB Benefit,Yes,Yes,No,No,111,s3://637423524537-eedb015-landing/raw/health_insurance/raw/2014/Benefits_Cost_Sharing_PUF.csv,2026-04-04 15:50:07.877589+00:00
4,2014,AK,21989,HIOS,6,2014-03-19 07:06:49,21989,AK,21989AK0010001,21989AK0010001-00,Major Dental Care - Child,No Charge,<NA>,No Charge,50%,<NA>,50%,Yes,<NA>,Covered,<NA>,<NA>,<NA>,<NA>,<NA>,See policy for additional limitations,Substantially Equal,Yes,Yes,No,No,112,s3://637423524537-eedb015-landing/raw/health_insurance/raw/2014/Benefits_Cost_Sharing_PUF.csv,2026-04-04 15:50:07.877589+00:00
5,2014,AK,21989,HIOS,6,2014-03-19 07:06:49,21989,AK,21989AK0010001,21989AK0010001-00,Basic Dental Care - Adult,No Charge,<NA>,No Charge,40%,<NA>,40%,<NA>,<NA>,Covered,Yes,1000,Dollars per Year,<NA>,<NA>,"Combined annual benefit maximum of $1000 per year for adults, 6 month waiting period, See policy...",Above EHB,Yes,Yes,No,No,113,s3://637423524537-eedb015-landing/raw/health_insurance/raw/2014/Benefits_Cost_Sharing_PUF.csv,2026-04-04 15:50:07.877589+00:00
6,2014,AK,21989,HIOS,6,2014-03-19 07:06:49,21989,AK,21989AK0010001,21989AK0010001-00,Orthodontia - Adult,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,114,s3://637423524537-eedb015-landing/raw/health_insurance/raw/2014/Benefits_Cost_Sharing_PUF.csv,2026-04-04 15:50:07.877589+00:00
7,2014,AK,21989,HIOS,6,2014-03-19 07:06:49,21989,AK,21989AK0010001,21989AK0010001-00,Major Dental Care - Adult,No Charge,<NA>,No Charge,50%,<NA>,50%,<NA>,<NA>,Covered,Yes,1000,Dollars per Year,<NA>,<NA>,"Combined annual benefit maximum of $1000 per year for adults, 12 month waiting period, See polic...",Above EHB,Yes,Yes,No,No,115,s3://637423524537-eedb015-landing/raw/health_insurance/raw/2014/Benefits_Cost_Sharing_PUF.csv,2026-04-04 15:50:07.877589+00:00
8,2014,AK,21989,HIOS,6,2014-03-19 07:06:49,21989,AK,21989AK0010001,21989AK0010001-00,Accidental Dental,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,118,s3://637423524537-eedb

...

,businessyear,statecode,issuerid,sourcename,versionnum,importdate,issuerid2,statecode2,standardcomponentid,planid,benefitname,copayinntier1,copayinntier2,copayoutofnet,coinsinntier1,coinsinntier2,coinsoutofnet,isehb,isstatemandate,iscovered,quantlimitonsvc,limitqty,limitunit,minimumstay,exclusions,explanation,ehbvarreason,issubjtodedtier1,issubjtodedtier2,isexclfrominnmoop,isexclfromoonmoop,rownumber,landinzone_path,ingestion_datetime
0,2015,AZ,12303,HIOS,5,9/3/2014 4:28,12303,AZ,12303AZ0010003,12303AZ0010003-00,Accidental Dental,No Charge,<NA>,No Charge,0%,<NA>,0%,<NA>,<NA>,Covered,Yes,1,Treatment(s) per Episode,<NA>,<NA>,"limit of service varies based upon procedure, see summary of benefits for additional information",Above EHB,No,No,No,Yes,118,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Benefits_Cost_Sharing_PUF.csv,2026-04-04 15:52:02.717819+00:00
1,2015,AZ,12303,HIOS,5,9/3/2014 4:28,12303,AZ,12303AZ0010003,12303AZ0010003-00,Basic Dental Care - Adult,No Charge after deductible,<NA>,No Charge after deductible,20%,<NA>,20%,<NA>,<NA>,Covered,No,<NA>,<NA>,<NA>,<NA>,<NA>,Above EHB,Yes,No,No,Yes,113,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Benefits_Cost_Sharing_PUF.csv,2026-04-04 15:52:02.717819+00:00
2,2015,AZ,12303,HIOS,5,9/3/2014 4:28,12303,AZ,12303AZ0010003,12303AZ0010003-00,Basic Dental Care - Child,No Charge after deductible,<NA>,No Charge after deductible,20%,<NA>,20%,<NA>,<NA>,Covered,No,<NA>,<NA>,<NA>,<NA>,<NA>,Above EHB,Yes,No,No,Yes,110,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Benefits_Cost_Sharing_PUF.csv,2026-04-04 15:52:02.717819+00:00
3,2015,AZ,12303,HIOS,5,9/3/2014 4:28,12303,AZ,12303AZ0010003,12303AZ0010003-00,Dental Check-Up for Children,No Charge,<NA>,No Charge,0%,<NA>,0%,Yes,<NA>,Covered,Yes,2,Visit(s) per Year,<NA>,<NA>,<NA>,Substantially Equal,No,No,No,Yes,104,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Benefits_Cost_Sharing_PUF.csv,2026-04-04 15:52:02.717819+00:00
4,2015,AZ,12303,HIOS,5,9/3/2014 4:28,12303,AZ,12303AZ0010003,12303AZ0010003-00,Major Dental Care - Adult,No Charge after deductible,<NA>,No Charge after deductible,50%,<NA>,50%,<NA>,<NA>,Covered,Yes,1,Item(s) per 3 Years,<NA>,<NA>,"limit of service varies based upon procedure, see summary of benefits for additional information",Above EHB,Yes,No,No,Yes,115,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Benefits_Cost_Sharing_PUF.csv,2026-04-04 15:52:02.717819+00:00
5,2015,AZ,12303,HIOS,5,9/3/2014 4:28,12303,AZ,12303AZ0010003,12303AZ0010003-00,Major Dental Care - Child,No Charge after deductible,<NA>,No Charge after deductible,50%,<NA>,50%,<NA>,<NA>,Covered,Yes,1,Item(s) per 3 Years,<NA>,<NA>,"limit of service varies based upon procedure, see summary of benefits for additional information",Substantially Equal,Yes,No,No,Yes,112,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Benefits_Cost_Sharing_PUF.csv,2026-04-04 15:52:02.717819+00:00
6,2015,AZ,12303,HIOS,5,9/3/2014 4:28,12303,AZ,12303AZ0010003,12303AZ0010003-00,Orthodontia - Adult,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,Not Covered,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,114,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Benefits_Cost_Sharing_PUF.csv,2026-04-04 15:52:02.717819+00:00
7,2015,AZ,12303,HIOS,5,9/3/2014 4:28,12303,AZ,12303AZ0010003,12303AZ0010003-00,Orthodontia - Child,No Charge,<NA>,No Charge,50%,<NA>,50%,<NA>,<NA>,Covered,Yes,1,Item(s) per Lifetime,<NA>,<NA>,<NA>,Substantially Equal,No,No,No,Yes,111,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Benefits_Cost_Sharing_PUF.csv,2026-04-04 15:52:02.717819+00:00
8,2015,AZ,12303,HIOS,5,9/3/2014 4:28,12303,AZ,12303AZ0010003,12303AZ0010003-00,Routine Dental Services (Adult),No Charge,<NA>,No Charge,0%,<NA>,0%,<NA>,<NA>,Covered,Yes,2,Visit(s) per Year,<NA>,<NA>,<NA>,Above EHB,No,No,No,Yes,68,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Benefits_Cost_Sharing_PUF.csv,2026-04-04 15:52:02.717819+00:00
9,2015

   ↳ 34 colunas | 10 linhas exibidas



### `raw_benefits_cost_sharing_puf_2015_12_08`

,businessyear,statecode,issuerid,sourcename,versionnum,importdate,issuerid2,statecode2,standardcomponentid,planid,benefitname,copayinntier1,copayinntier2,copayoutofnet,coinsinntier1,coinsinntier2,coinsoutofnet,isehb,isstatemandate,iscovered,quantlimitonsvc,limitqty,limitunit,minimumstay,exclusions,explanation,ehbvarreason,isexclfrominnmoop,isexclfromoonmoop,rownumber,landinzone_path,ingestion_datetime
0,2016,AK,21989,HIOS,4,2015-08-22 15:09:32,21989,AK,21989AK0030001,21989AK0030001-00,Routine Dental Services (Adult),Not Applicable,<NA>,Not Applicable,20% Coinsurance after deductible,<NA>,20% Coinsurance after deductible,<NA>,<NA>,Covered,Yes,1000,Dollars per Year,<NA>,<NA>,Combined annual benefit maximum,Above EHB,No,No,68,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Benefits_Cost_Sharing_PUF_2015-12-08.csv,2026-04-04 15:52:15.884875+00:00
1,2016,AK,21989,HIOS,4,2015-08-22 15:09:32,21989,AK,21989AK0030001,21989AK0030001-00,Dental Check-Up for Children,Not Applicable,<NA>,Not Applicable,20% Coinsurance after deductible,<NA>,20% Coinsurance after deductible,Yes,<NA>,Covered,Yes,1,Visit(s) per 6 Months,<NA>,<NA>,<NA>,Additional EHB Benefit,No,No,104,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Benefits_Cost_Sharing_PUF_2015-12-08.csv,2026-04-04 15:52:15.884875+00:00
2,2016,AK,21989,HIOS,4,2015-08-22 15:09:32,21989,AK,21989AK0030001,21989AK0030001-00,Basic Dental Care - Child,Not Applicable,<NA>,Not Applicable,30% Coinsurance after deductible,<NA>,30% Coinsurance after deductible,Yes,<NA>,Covered,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,Additional EHB Benefit,No,No,110,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Benefits_Cost_Sharing_PUF_2015-12-08.csv,2026-04-04 15:52:15.884875+00:00
3,2016,AK,21989,HIOS,4,2015-08-22 15:09:32,21989,AK,21989AK0030001,21989AK0030001-00,Orthodontia - Child,Not Applicable,<NA>,Not Applicable,50% Coinsurance after deductible,<NA>,50% Coinsurance after deductible,Yes,<NA>,Covered,<NA>,<NA>,<NA>,<NA>,<NA>,24 month waiting period,Additional EHB Benefit,No,No,111,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Benefits_Cost_Sharing_PUF_2015-12-08.csv,2026-04-04 15:52:15.884875+00:00
4,2016,AK,21989,HIOS,4,2015-08-22 15:09:32,21989,AK,21989AK0030001,21989AK0030001-00,Major Dental Care - Child,Not Applicable,<NA>,Not Applicable,50% Coinsurance after deductible,<NA>,50% Coinsurance after deductible,Yes,<NA>,Covered,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,Additional EHB Benefit,No,No,112,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Benefits_Cost_Sharing_PUF_2015-12-08.csv,2026-04-04 15:52:15.884875+00:00
5,2016,AK,21989,HIOS,4,2015-08-22 15:09:32,21989,AK,21989AK0030001,21989AK0030001-00,Basic Dental Care - Adult,Not Applicable,<NA>,Not Applicable,30% Coinsurance after deductible,<NA>,30% Coinsurance after deductible,<NA>,<NA>,Covered,Yes,1000,Dollars per Year,<NA>,<NA>,6 mo waiting period for age 19 and over if no prior coverage within last 12 months with no more ...,Above EHB,No,No,113,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Benefits_Cost_Sharing_PUF_2015-12-08.csv,2026-04-04 15:52:15.884875+00:00
6,2016,AK,21989,HIOS,4,2015-08-22 15:09:32,21989,AK,21989AK0030001,21989AK0030001-00,Orthodontia - Adult,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,114,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Benefits_Cost_Sharing_PUF_2015-12-08.csv,2026-04-04 15:52:15.884875+00:00
7,2016,AK,21989,HIOS,4,2015-08-22 15:09:32,21989,AK,21989AK0030001,21989AK0030001-00,Major Dental Care - Adult,Not Applicable,<NA>,Not Applicable,50% Coinsurance after deductible,<NA>,50% Coinsurance after deductible,<NA>,<NA>,Covered,Yes,1000,Dollars per Year,<NA>,<NA>,12 mo waiting period for age 19 and over if no prior coverage within last 12 months with no more...,Above EHB,No,No,115,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Benefits_Cost_Sharing_PUF_2015-12-08.csv,2026-04-04 15:52:15.884875+00:00
8,201

   ↳ 32 colunas | 10 linhas exibidas



---
## 6. Tabela: ServiceArea — Áreas de Serviço

**O que é:** Define as **regiões geográficas** onde cada seguradora está autorizada a oferecer planos. A granularidade pode ser estado inteiro, condados específicos ou até sub-divisões de condados (partial county / ZIP codes). É o arquivo que permite mapear "desertos de cobertura".

**Colunas principais:**

| Coluna | Tipo | Descrição |
|---|---|---|
| `BusinessYear` | integer | Ano |
| `StateCode` | string | Estado |
| `IssuerId` | string | Código da seguradora |
| `SourceName` | string | Fonte/exchange |
| `VersionNum` | integer | Versão da submissão |
| `ImportDate` | date | Data de importação |
| `IssuerId2` | string | ID alternativo |
| `FederalTIN` | string | TIN federal |
| `ServiceAreaId` | string | Identificador único da área de serviço (chave para `PlanAttributes`) |
| `ServiceAreaName` | string | Nome descritivo da área (ex: "California Statewide") |
| `CoverEntireState` | string | Cobre o estado inteiro? (`Yes`/`No`) |
| `County` | string | Nome do condado coberto (quando não cobre o estado todo) |
| `CountyCode` | string | Código FIPS do condado |
| `PartialCounty` | string | Cobertura parcial de condado? (`Yes`/`No`) |
| `ZipCodes` | string | Lista de CEPs cobertos (quando a cobertura é por ZIP) |
| `PartialCountyJustification` | string | Justificativa regulatória para cobertura parcial |

**Relações:** `ServiceAreaId` → `PlanAttributes.ServiceAreaId`; `IssuerId` → `PlanAttributes.IssuerId`.

**Insights dos notebooks Kaggle:**
- Seguradoras com `CoverEntireState = False` e poucos condados cobertos tendem a ter redes menores e preços mais baixos na tabela `Rate`.
- Condados cobertos por apenas 1 seguradora são candidatos a efeito de monopólio — preços artificialmente inflados em comparação com condados competitivos.
- A granularidade de `ZipCodes` permite cruzar com dados populacionais do Census para identificar "desertos de cobertura" em regiões rurais.

**Relevância para o projeto:** Essencial para as **Questões 4 e 5** — determina a amplitude geográfica das redes e identifica regiões com monopólio ou baixa competição.

In [8]:
show_family(["service_area"])

### `raw_2014_service_area_puf`

,businessyear,statecode,issuerid,sourcename,versionnum,importdate,issuerid2,statecode2,serviceareaid,serviceareaname,coverentirestate,county,partialcounty,zipcodes,partialcountyjustification,rownumber,marketcoverage,dentalonly,landinzone_path,ingestion_datetime
0,2014,PA,22444,HIOS,9,2014-01-21 08:29:49,22444,PA,PAS001,Geisinger Health Plan,No,42103,No,<NA>,<NA>,42,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/raw/2014/Service_Area_PUF.csv,2026-04-04 15:50:40.345562+00:00
1,2014,PA,22444,HIOS,9,2014-01-21 08:29:49,22444,PA,PAS001,Geisinger Health Plan,No,42105,No,<NA>,<NA>,43,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/raw/2014/Service_Area_PUF.csv,2026-04-04 15:50:40.345562+00:00
2,2014,PA,22444,HIOS,9,2014-01-21 08:29:49,22444,PA,PAS001,Geisinger Health Plan,No,42107,No,<NA>,<NA>,44,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/raw/2014/Service_Area_PUF.csv,2026-04-04 15:50:40.345562+00:00
3,2014,PA,22444,HIOS,9,2014-01-21 08:29:49,22444,PA,PAS001,Geisinger Health Plan,No,42109,No,<NA>,<NA>,45,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/raw/2014/Service_Area_PUF.csv,2026-04-04 15:50:40.345562+00:00
4,2014,PA,22444,HIOS,9,2014-01-21 08:29:49,22444,PA,PAS001,Geisinger Health Plan,No,42111,No,<NA>,<NA>,46,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/raw/2014/Service_Area_PUF.csv,2026-04-04 15:50:40.345562+00:00
5,2014,PA,22444,HIOS,9,2014-01-21 08:29:49,22444,PA,PAS001,Geisinger Health Plan,No,42113,No,<NA>,<NA>,47,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/raw/2014/Service_Area_PUF.csv,2026-04-04 15:50:40.345562+00:00
6,2014,PA,22444,HIOS,9,2014-01-21 08:29:49,22444,PA,PAS001,Geisinger Health Plan,No,42115,No,<NA>,<NA>,48,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/raw/2014/Service_Area_PUF.csv,2026-04-04 15:50:40.345562+00:00
7,2014,PA,22444,HIOS,9,2014-01-21 08:29:49,22444,PA,PAS001,Geisinger Health Plan,No,42117,No,<NA>,<NA>,49,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/raw/2014/Service_Area_PUF.csv,2026-04-04 15:50:40.345562+00:00
8,2014,PA,22444,HIOS,9,2014-01-21 08:29:49,22444,PA,PAS001,Geisinger Health Plan,No,42119,No,<NA>,<NA>,50,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/raw/2014/Service_Area_PUF.csv,2026-04-04 15:50:40.345562+00:00
9,2014,PA,22444,HIOS,9,2014-01-21 08:29:49,22444,PA,PAS001,Geisinger Health Plan,No,42127,No,<NA>,<NA>,51,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/raw/2014/Service_Area_PUF.csv,2026-04-04 15:50:40.345562+00:00


   ↳ 20 colunas | 10 linhas exibidas



### `raw_2015_service_area_puf`

,businessyear,statecode,issuerid,sourcename,versionnum,importdate,issuerid2,statecode2,serviceareaid,serviceareaname,coverentirestate,county,partialcounty,zipcodes,partialcountyjustification,rownumber,marketcoverage,dentalonly,landinzone_path,ingestion_datetime
0,2015,PA,70406,HIOS,2,2014-08-05 13:28:44,70406,PA,PAS001,Pennsylvania,No,42049,No,<NA>,<NA>,35,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/raw/2015/Service_Area_PUF.csv,2026-04-04 15:51:23.545236+00:00
1,2015,PA,70406,HIOS,2,2014-08-05 13:28:44,70406,PA,PAS001,Pennsylvania,No,42051,No,<NA>,<NA>,36,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/raw/2015/Service_Area_PUF.csv,2026-04-04 15:51:23.545236+00:00
2,2015,PA,70406,HIOS,2,2014-08-05 13:28:44,70406,PA,PAS001,Pennsylvania,No,42055,No,<NA>,<NA>,37,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/raw/2015/Service_Area_PUF.csv,2026-04-04 15:51:23.545236+00:00
3,2015,PA,70406,HIOS,2,2014-08-05 13:28:44,70406,PA,PAS001,Pennsylvania,No,42059,No,<NA>,<NA>,38,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/raw/2015/Service_Area_PUF.csv,2026-04-04 15:51:23.545236+00:00
4,2015,PA,70406,HIOS,2,2014-08-05 13:28:44,70406,PA,PAS001,Pennsylvania,No,42061,No,<NA>,<NA>,39,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/raw/2015/Service_Area_PUF.csv,2026-04-04 15:51:23.545236+00:00
5,2015,PA,70406,HIOS,2,2014-08-05 13:28:44,70406,PA,PAS001,Pennsylvania,No,42063,No,<NA>,<NA>,40,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/raw/2015/Service_Area_PUF.csv,2026-04-04 15:51:23.545236+00:00
6,2015,PA,70406,HIOS,2,2014-08-05 13:28:44,70406,PA,PAS001,Pennsylvania,No,42065,No,<NA>,<NA>,41,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/raw/2015/Service_Area_PUF.csv,2026-04-04 15:51:23.545236+00:00
7,2015,PA,70406,HIOS,2,2014-08-05 13:28:44,70406,PA,PAS001,Pennsylvania,No,42067,No,<NA>,<NA>,42,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/raw/2015/Service_Area_PUF.csv,2026-04-04 15:51:23.545236+00:00
8,2015,PA,70406,HIOS,2,2014-08-05 13:28:44,70406,PA,PAS001,Pennsylvania,No,42069,No,<NA>,<NA>,43,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/raw/2015/Service_Area_PUF.csv,2026-04-04 15:51:23.545236+00:00
9,2015,PA,70406,HIOS,2,2014-08-05 13:28:44,70406,PA,PAS001,Pennsylvania,No,42071,No,<NA>,<NA>,44,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/raw/2015/Service_Area_PUF.csv,2026-04-04 15:51:23.545236+00:00


...

,businessyear,statecode,issuerid,sourcename,versionnum,importdate,issuerid2,statecode2,serviceareaid,serviceareaname,coverentirestate,county,partialcounty,zipcodes,partialcountyjustification,rownumber,marketcoverage,dentalonlyplan,landinzone_path,ingestion_datetime
0,2016,NJ,10191,HIOS,6,2015-11-18 07:25:09,10191,NJ,NJS003,Monmouth/Centra State,No,34029,Yes,"08527, 08533, 08701",10191-Partial County-NJS003-Monmouth.pdf,18,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/raw/ServiceArea_PUF_2015-12-08.csv,2026-04-04 15:53:23.722664+00:00
1,2016,NJ,10191,HIOS,6,2015-11-18 07:25:09,10191,NJ,NJS004,PCMH,No,34001,No,<NA>,<NA>,19,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/raw/ServiceArea_PUF_2015-12-08.csv,2026-04-04 15:53:23.722664+00:00
2,2016,NJ,10191,HIOS,6,2015-11-18 07:25:09,10191,NJ,NJS004,PCMH,No,34003,No,<NA>,<NA>,20,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/raw/ServiceArea_PUF_2015-12-08.csv,2026-04-04 15:53:23.722664+00:00
3,2016,NJ,10191,HIOS,6,2015-11-18 07:25:09,10191,NJ,NJS004,PCMH,No,34005,No,<NA>,<NA>,21,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/raw/ServiceArea_PUF_2015-12-08.csv,2026-04-04 15:53:23.722664+00:00
4,2016,NJ,10191,HIOS,6,2015-11-18 07:25:09,10191,NJ,NJS004,PCMH,No,34007,No,<NA>,<NA>,22,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/raw/ServiceArea_PUF_2015-12-08.csv,2026-04-04 15:53:23.722664+00:00
5,2016,NJ,10191,HIOS,6,2015-11-18 07:25:09,10191,NJ,NJS004,PCMH,No,34009,No,<NA>,<NA>,23,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/raw/ServiceArea_PUF_2015-12-08.csv,2026-04-04 15:53:23.722664+00:00
6,2016,NJ,10191,HIOS,6,2015-11-18 07:25:09,10191,NJ,NJS004,PCMH,No,34011,No,<NA>,<NA>,24,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/raw/ServiceArea_PUF_2015-12-08.csv,2026-04-04 15:53:23.722664+00:00
7,2016,NJ,10191,HIOS,6,2015-11-18 07:25:09,10191,NJ,NJS004,PCMH,No,34013,No,<NA>,<NA>,25,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/raw/ServiceArea_PUF_2015-12-08.csv,2026-04-04 15:53:23.722664+00:00
8,2016,NJ,10191,HIOS,6,2015-11-18 07:25:09,10191,NJ,NJS004,PCMH,No,34015,No,<NA>,<NA>,26,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/raw/ServiceArea_PUF_2015-12-08.csv,2026-04-04 15:53:23.722664+00:00
9,2016,NJ,10191,HIOS,6,2015-11-18 07:25:09,10191,NJ,NJS004,PCMH,No,34017,No,<NA>,<NA>,27,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/raw/ServiceArea_PUF_2015-12-08.csv,2026-04-04 15:53:23.722664+00:00


   ↳ 20 colunas | 10 linhas exibidas



### `service_area`

,businessyear,statecode,issuerid,sourcename,versionnum,importdate,issuerid2,statecode2,serviceareaid,serviceareaname,coverentirestate,county,partialcounty,zipcodes,partialcountyjustification,rownumber,marketcoverage,dentalonlyplan,landinzone_path,ingestion_datetime
0,2014,PA,22444,HIOS,9,2014-01-21 08:29:49,22444,PA,PAS001,Geisinger Health Plan,No,42103.0,No,<NA>,<NA>,42,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/ServiceArea.csv,2026-04-04 15:53:25.295481+00:00
1,2014,PA,22444,HIOS,9,2014-01-21 08:29:49,22444,PA,PAS001,Geisinger Health Plan,No,42105.0,No,<NA>,<NA>,43,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/ServiceArea.csv,2026-04-04 15:53:25.295481+00:00
2,2014,PA,22444,HIOS,9,2014-01-21 08:29:49,22444,PA,PAS001,Geisinger Health Plan,No,42107.0,No,<NA>,<NA>,44,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/ServiceArea.csv,2026-04-04 15:53:25.295481+00:00
3,2014,PA,22444,HIOS,9,2014-01-21 08:29:49,22444,PA,PAS001,Geisinger Health Plan,No,42109.0,No,<NA>,<NA>,45,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/ServiceArea.csv,2026-04-04 15:53:25.295481+00:00
4,2014,PA,22444,HIOS,9,2014-01-21 08:29:49,22444,PA,PAS001,Geisinger Health Plan,No,42111.0,No,<NA>,<NA>,46,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/ServiceArea.csv,2026-04-04 15:53:25.295481+00:00
5,2014,PA,22444,HIOS,9,2014-01-21 08:29:49,22444,PA,PAS001,Geisinger Health Plan,No,42113.0,No,<NA>,<NA>,47,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/ServiceArea.csv,2026-04-04 15:53:25.295481+00:00
6,2014,PA,22444,HIOS,9,2014-01-21 08:29:49,22444,PA,PAS001,Geisinger Health Plan,No,42115.0,No,<NA>,<NA>,48,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/ServiceArea.csv,2026-04-04 15:53:25.295481+00:00
7,2014,PA,22444,HIOS,9,2014-01-21 08:29:49,22444,PA,PAS001,Geisinger Health Plan,No,42117.0,No,<NA>,<NA>,49,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/ServiceArea.csv,2026-04-04 15:53:25.295481+00:00
8,2014,PA,22444,HIOS,9,2014-01-21 08:29:49,22444,PA,PAS001,Geisinger Health Plan,No,42119.0,No,<NA>,<NA>,50,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/ServiceArea.csv,2026-04-04 15:53:25.295481+00:00
9,2014,PA,22444,HIOS,9,2014-01-21 08:29:49,22444,PA,PAS001,Geisinger Health Plan,No,42127.0,No,<NA>,<NA>,51,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/ServiceArea.csv,2026-04-04 15:53:25.295481+00:00


   ↳ 20 colunas | 10 linhas exibidas



---
## 7. Tabela: BusinessRules — Regras de Negócio dos Planos

**O que é:** Define as **regras operacionais e de elegibilidade** de cada plano: idades mínima/máxima do segurado, períodos de carência, regras de pagamento e penalidades. Complementa `PlanAttributes` com informações mais operacionais — é a tabela menos explorada nas análises públicas do Kaggle.

**Colunas principais:**

| Coluna | Tipo | Descrição |
|---|---|---|
| `BusinessYear` | integer | Ano |
| `StateCode` | string | Estado |
| `IssuerId` | string | Código da seguradora |
| `PlanId` | string | ID do plano |
| `EnrolleeMinimumAge` | integer | Idade mínima do segurado |
| `EnrolleeMaximumAge` | integer | Idade máxima do segurado |
| `PlanEffectiveDate` | date | Data de início de vigência do plano |
| `PlanExpirationDate` | date | Data de expiração do plano |
| `GracePeriodDetail` | string | Detalhes do período de carência para pagamento |
| `MarketCoverage` | string | Tipo de mercado: `Individual` / `Small Group` |
| `DentalOnlyPlan` | string | Plano exclusivamente odontológico (`Yes`/`No`) |
| `SubjectToMedicalManagement` | string | Sujeito a gerenciamento médico prévio |
| `FirstTierUtilization` | decimal | % de utilização de serviços de primeiro nível |
| `SecondTierUtilization` | decimal | % de utilização de serviços de segundo nível |
| `InpatientCopaymentMaximumDays` | integer | Máximo de dias com copagamento em internação |
| `BeginPrimaryCareCostSharingAfterNumberOfVisits` | integer | Visitas gratuitas antes de ativar o cost-sharing |
| `BeginPrimaryCareDeductibleCoinsuranceAfterNumberOfCopays` | integer | Copays antes de ativar o deductible/coinsurance |

**Relações:** `PlanId` → `PlanAttributes.PlanId`; `IssuerId` → `PlanAttributes.IssuerId`.

**Relevância para o projeto:** Útil para filtrar planos elegíveis para os perfis de beneficiário analisados nas questões e para entender como `BeginPrimaryCareCostSharingAfterNumberOfVisits` impacta o custo total em tratamentos recorrentes (Questão 1 — infusão crônica).

In [9]:
show_family(["business_rules"])

### `business_rules`

,businessyear,statecode,issuerid,sourcename,versionnum,importdate,issuerid2,tin,productid,standardcomponentid,enrolleecontractratedeterminationrule,twoparentfamilymaxdependentsrule,singleparentfamilymaxdependentsrule,dependentmaximumagrule,childrenonlycontractmaxchildrenrule,domesticpartnerasspouseindicator,samesexpartnerasspouseindicator,agedeterminationrule,minimumtobaccofreemonthsrule,cohabitationrule,rownumber,marketcoverage,dentalonlyplan,landinzone_path,ingestion_datetime
0,2014,AL,82285,HIOS,7,2014-01-21 08:29:49,82285,94-2761537,<NA>,82285AL0010006,A different rate (specifically for parties of two or more)for each enrollee is added together,3 or more,3 or more,26,3 or more,Yes,Yes,Age on effective date,Not Applicable,"Spouse,No;Adopted Child,No;Foster Child,No;Ward,No;Stepson or Stepdaughter,No;Child,No;Life Part...",14,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/BusinessRules.csv,2026-04-04 15:49:05.100408+00:00
1,2014,AL,82285,HIOS,7,2014-01-21 08:29:49,82285,94-2761537,<NA>,82285AL0020004,A different rate (specifically for parties of two or more)for each enrollee is added together,3 or more,3 or more,26,3 or more,Yes,Yes,Age on effective date,Not Applicable,"Spouse,No;Adopted Child,No;Foster Child,No;Ward,No;Stepson or Stepdaughter,No;Child,No;Life Part...",17,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/BusinessRules.csv,2026-04-04 15:49:05.100408+00:00
2,2014,AL,82285,HIOS,7,2014-01-21 08:29:49,82285,94-2761537,<NA>,82285AL0020006,A different rate (specifically for parties of two or more)for each enrollee is added together,3 or more,3 or more,26,3 or more,Yes,Yes,Age on effective date,Not Applicable,"Spouse,No;Adopted Child,No;Foster Child,No;Ward,No;Stepson or Stepdaughter,No;Child,No;Life Part...",18,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/BusinessRules.csv,2026-04-04 15:49:05.100408+00:00
3,2014,AZ,17100,HIOS,7,2013-10-15 07:27:56,17100,13-5581829,<NA>,<NA>,There are rates specifically for couples and for families (not just addition of individual rates),3 or more,3 or more,26,3 or more,Yes,Yes,Age on effective date,Not Applicable,"Spouse,No;Grandson or Granddaughter,Yes;Adopted Child,Yes;Foster Child,Yes;Ward,Yes;Stepson or S...",10,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/BusinessRules.csv,2026-04-04 15:49:05.100408+00:00
4,2014,AZ,17100,HIOS,7,2013-10-15 07:27:56,17100,13-5581829,<NA>,17100AZ0160001,A different rate (specifically for parties of two or more)for each enrollee is added together,3 or more,3 or more,21,3 or more,No,No,Age on effective date,Not Applicable,"Brother or Sister,Yes;Ward,Yes;Self,Yes;Child,Yes;Dependent on a Minor Dependent,Yes",11,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/BusinessRules.csv,2026-04-04 15:49:05.100408+00:00
5,2014,AL,82285,HIOS,7,2014-01-21 08:29:49,82285,94-2761537,<NA>,82285AL0020001,A different rate (specifically for parties of two or more)for each enrollee is added together,3 or more,3 or more,Not Applicable,3 or more,Yes,Yes,Age on effective date,Not Applicable,"Adopted Child,Yes;Foster Child,Yes;Stepson or Stepdaughter,Yes;Child,Yes",15,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/BusinessRules.csv,2026-04-04 15:49:05.100408+00:00
6,2014,AL,82285,HIOS,7,2014-01-21 08:29:49,82285,94-2761537,<NA>,82285AL0020002,A different rate (specifically for parties of two or more)for each enrollee is added together,3 or more,3 or more,Not Applicable,3 or more,Yes,Yes,Age on effective date,Not Applicable,"Adopted Child,Yes;Foster Child,Yes;Stepson or Stepdaughter,Yes;Child,Yes",16,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/BusinessRules.csv,2026-04-04 15:49:05.100408+00:00
7,2014,AZ,18156,HIOS,1,2013-06-06 10:50:48,18156,35-0472300,<NA>,<NA>,There are rates specifically for couples and for families (not just addition of individual rates),3 or more,3 or more,30,1,Yes,Yes,Age on effective date,Not Applicable,"Spouse,No;Grandson o

   ↳ 25 colunas | 10 linhas exibidas



### `raw_2014_business_rules_puf`

,businessyear,statecode,issuerid,sourcename,versionnum,importdate,issuerid2,tin,productid,standardcomponentid,enrolleecontractratedeterminationrule,twoparentfamilymaxdependentsrule,singleparentfamilymaxdependentsrule,dependentmaximumagrule,childrenonlycontractmaxchildrenrule,domesticpartnerasspouseindicator,samesexpartnerasspouseindicator,agedeterminationrule,minimumtobaccofreemonthsrule,cohabitationrule,rownumber,marketcoverage,dentalonly,landinzone_path,ingestion_datetime
0,2014,AL,82285,HIOS,7,2014-01-21 08:29:49,82285,94-2761537,<NA>,82285AL0010006,A different rate (specifically for parties of two or more)for each enrollee is added together,3 or more,3 or more,26,3 or more,Yes,Yes,Age on effective date,Not Applicable,"Spouse,No;Adopted Child,No;Foster Child,No;Ward,No;Stepson or Stepdaughter,No;Child,No;Life Part...",14,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/raw/2014/Business_Rules_PUF.csv,2026-04-04 15:50:16.093177+00:00
1,2014,AL,82285,HIOS,7,2014-01-21 08:29:49,82285,94-2761537,<NA>,82285AL0020001,A different rate (specifically for parties of two or more)for each enrollee is added together,3 or more,3 or more,Not Applicable,3 or more,Yes,Yes,Age on effective date,Not Applicable,"Adopted Child,Yes;Foster Child,Yes;Stepson or Stepdaughter,Yes;Child,Yes",15,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/raw/2014/Business_Rules_PUF.csv,2026-04-04 15:50:16.093177+00:00
2,2014,AL,82285,HIOS,7,2014-01-21 08:29:49,82285,94-2761537,<NA>,82285AL0020002,A different rate (specifically for parties of two or more)for each enrollee is added together,3 or more,3 or more,Not Applicable,3 or more,Yes,Yes,Age on effective date,Not Applicable,"Adopted Child,Yes;Foster Child,Yes;Stepson or Stepdaughter,Yes;Child,Yes",16,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/raw/2014/Business_Rules_PUF.csv,2026-04-04 15:50:16.093177+00:00
3,2014,AL,82285,HIOS,7,2014-01-21 08:29:49,82285,94-2761537,<NA>,82285AL0020004,A different rate (specifically for parties of two or more)for each enrollee is added together,3 or more,3 or more,26,3 or more,Yes,Yes,Age on effective date,Not Applicable,"Spouse,No;Adopted Child,No;Foster Child,No;Ward,No;Stepson or Stepdaughter,No;Child,No;Life Part...",17,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/raw/2014/Business_Rules_PUF.csv,2026-04-04 15:50:16.093177+00:00
4,2014,AL,82285,HIOS,7,2014-01-21 08:29:49,82285,94-2761537,<NA>,82285AL0020006,A different rate (specifically for parties of two or more)for each enrollee is added together,3 or more,3 or more,26,3 or more,Yes,Yes,Age on effective date,Not Applicable,"Spouse,No;Adopted Child,No;Foster Child,No;Ward,No;Stepson or Stepdaughter,No;Child,No;Life Part...",18,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/raw/2014/Business_Rules_PUF.csv,2026-04-04 15:50:16.093177+00:00
5,2014,AZ,17100,HIOS,7,2013-10-15 07:27:56,17100,13-5581829,<NA>,<NA>,There are rates specifically for couples and for families (not just addition of individual rates),3 or more,3 or more,26,3 or more,Yes,Yes,Age on effective date,Not Applicable,"Spouse,No;Grandson or Granddaughter,Yes;Adopted Child,Yes;Foster Child,Yes;Ward,Yes;Stepson or S...",10,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/raw/2014/Business_Rules_PUF.csv,2026-04-04 15:50:16.093177+00:00
6,2014,AZ,17100,HIOS,7,2013-10-15 07:27:56,17100,13-5581829,<NA>,17100AZ0160001,A different rate (specifically for parties of two or more)for each enrollee is added together,3 or more,3 or more,21,3 or more,No,No,Age on effective date,Not Applicable,"Brother or Sister,Yes;Ward,Yes;Self,Yes;Child,Yes;Dependent on a Minor Dependent,Yes",11,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/raw/2014/Business_Rules_PUF.csv,2026-04-04 15:50:16.093177+00:00
7,2014,AZ,18156,HIOS,1,2013-06-06 10:50:48,18156,35-0472300,<NA>,<NA>,There are rates specifically for couples and for families (not just addition of individual rate

...

,businessyear,statecode,issuerid,sourcename,versionnum,importdate,issuerid2,tin,productid,standardcomponentid,enrolleecontractratedeterminationrule,twoparentfamilymaxdependentsrule,singleparentfamilymaxdependentsrule,dependentmaximumagrule,childrenonlycontractmaxchildrenrule,domesticpartnerasspouseindicator,samesexpartnerasspouseindicator,agedeterminationrule,minimumtobaccofreemonthsrule,cohabitationrule,rownumber,marketcoverage,dentalonlyplan,landinzone_path,ingestion_datetime
0,2016,AK,21989,HIOS,4,2015-08-22 15:09:32,21989,93-0438772,21989AK009,21989AK0090001,There are rates specifically for couples and for families (not just addition of individual rates),3 or more,3 or more,25,3 or more,Yes,Yes,Age on effective date,Not Applicable,"Spouse,Yes;Adopted Child,No;Stepson or Stepdaughter,No;Self,Yes;Child,No;Life Partner,Yes",15,SHOP (Small Group),Yes,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Business_Rules_PUF_2015-12-08.csv,2026-04-04 15:52:27.860490+00:00
1,2016,AK,21989,HIOS,4,2015-08-22 15:09:32,21989,93-0438772,21989AK010,21989AK0100002,There are rates specifically for couples and for families (not just addition of individual rates),3 or more,3 or more,25,3 or more,Yes,Yes,Age on effective date,Not Applicable,"Spouse,Yes;Adopted Child,No;Stepson or Stepdaughter,No;Self,Yes;Child,No;Life Partner,Yes",16,SHOP (Small Group),Yes,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Business_Rules_PUF_2015-12-08.csv,2026-04-04 15:52:27.860490+00:00
2,2016,AK,21989,HIOS,4,2015-08-22 15:09:32,21989,93-0438772,21989AK011,21989AK0110001,There are rates specifically for couples and for families (not just addition of individual rates),3 or more,3 or more,25,3 or more,Yes,Yes,Age on effective date,Not Applicable,"Spouse,Yes;Adopted Child,No;Stepson or Stepdaughter,No;Self,Yes;Child,No;Life Partner,Yes",17,SHOP (Small Group),Yes,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Business_Rules_PUF_2015-12-08.csv,2026-04-04 15:52:27.860490+00:00
3,2016,AK,21989,HIOS,4,2015-08-22 15:09:32,21989,93-0438772,21989AK012,21989AK0120001,There are rates specifically for couples and for families (not just addition of individual rates),3 or more,3 or more,25,3 or more,Yes,Yes,Age on effective date,Not Applicable,"Spouse,Yes;Adopted Child,No;Stepson or Stepdaughter,No;Self,Yes;Child,No;Life Partner,Yes",18,SHOP (Small Group),Yes,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Business_Rules_PUF_2015-12-08.csv,2026-04-04 15:52:27.860490+00:00
4,2016,AK,21989,HIOS,4,2015-08-22 15:09:32,21989,93-0438772,21989AK003,21989AK0030001,A different rate (specifically for parties of two or more)for each enrollee is added together,3 or more,3 or more,25,3 or more,Yes,Yes,Age on effective date,Not Applicable,"Spouse,Yes;Adopted Child,No;Stepson or Stepdaughter,No;Self,Yes;Child,No;Life Partner,Yes",11,Individual,Yes,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Business_Rules_PUF_2015-12-08.csv,2026-04-04 15:52:27.860490+00:00
5,2016,AK,21989,HIOS,4,2015-08-22 15:09:32,21989,93-0438772,21989AK009,21989AK0090002,There are rates specifically for couples and for families (not just addition of individual rates),3 or more,3 or more,25,3 or more,Yes,Yes,Age on effective date,Not Applicable,"Spouse,Yes;Adopted Child,No;Stepson or Stepdaughter,No;Self,Yes;Child,No;Life Partner,Yes",15,SHOP (Small Group),Yes,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Business_Rules_PUF_2015-12-08.csv,2026-04-04 15:52:27.860490+00:00
6,2016,AK,21989,HIOS,4,2015-08-22 15:09:32,21989,93-0438772,21989AK010,21989AK0100001,There are rates specifically for couples and for families (not just addition of individual rates),3 or more,3 or more,25,3 or more,Yes,Yes,Age on effective date,Not Applicable,"Spouse,Yes;Adopted Child,No;Stepson or Stepdaughter,No;Self,Yes;Child,No;Life Partner,Yes",16,SHOP (Small Group),Yes,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Business_Rules_PUF_2015-12-08.csv,2026-04-04 15:52:27.

   ↳ 25 colunas | 10 linhas exibidas



### `raw_business_rules_puf_reformat`

,businessyear,statecode,issuerid,sourcename,versionnum,importdate,issuerid2,tin,productid,standardcomponentid,enrolleecontractratedeterminationrule,twoparentfamilymaxdependentsrule,singleparentfamilymaxdependentsrule,dependentmaximumagrule,childrenonlycontractmaxchildrenrule,domesticpartnerasspouseindicator,samesexpartnerasspouseindicator,agedeterminationrule,minimumtobaccofreemonthsrule,cohabitationrule,rownumber,marketcoverage,dentalonly,landinzone_path,ingestion_datetime
0,2015,AK,73836,HIOS,15,2015-04-22 11:06:15,73836,93-0989307,73836AK073,73836AK0730003,A different rate (specifically for parties of two or more)for each enrollee is added together,3 or more,3 or more,25,3 or more,Yes,Yes,Age on effective date,Not Applicable,"Spouse,No;Adopted Child,No;Ward,No;Stepson or Stepdaughter,No;Self,Yes;Child,No;Dependent on a M...",10,SHOP (Small Group),No,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Business_Rules_PUF_Reformat.csv,2026-04-04 15:52:29.620548+00:00
1,2015,AK,73836,HIOS,15,2015-04-22 11:06:15,73836,93-0989307,73836AK073,73836AK0730006,A different rate (specifically for parties of two or more)for each enrollee is added together,3 or more,3 or more,25,3 or more,Yes,Yes,Age on effective date,Not Applicable,"Spouse,No;Adopted Child,No;Ward,No;Stepson or Stepdaughter,No;Self,Yes;Child,No;Dependent on a M...",10,SHOP (Small Group),No,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Business_Rules_PUF_Reformat.csv,2026-04-04 15:52:29.620548+00:00
2,2015,AK,73836,HIOS,15,2015-04-22 11:06:15,73836,93-0989307,73836AK074,73836AK0740001,A different rate (specifically for parties of two or more)for each enrollee is added together,3 or more,3 or more,25,3 or more,Yes,Yes,Age on effective date,Not Applicable,"Spouse,No;Adopted Child,No;Ward,No;Stepson or Stepdaughter,No;Self,Yes;Child,No;Dependent on a M...",10,SHOP (Small Group),No,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Business_Rules_PUF_Reformat.csv,2026-04-04 15:52:29.620548+00:00
3,2015,AK,73836,HIOS,15,2015-04-22 11:06:15,73836,93-0989307,73836AK074,73836AK0740002,A different rate (specifically for parties of two or more)for each enrollee is added together,3 or more,3 or more,25,3 or more,Yes,Yes,Age on effective date,Not Applicable,"Spouse,No;Adopted Child,No;Ward,No;Stepson or Stepdaughter,No;Self,Yes;Child,No;Dependent on a M...",10,SHOP (Small Group),No,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Business_Rules_PUF_Reformat.csv,2026-04-04 15:52:29.620548+00:00
4,2015,AL,82285,HIOS,8,2014-11-14 05:23:27,82285,94-2761537,82285AL002,82285AL0020001,A different rate (specifically for parties of two or more)for each enrollee is added together,3 or more,3 or more,Not Applicable,3 or more,Yes,Yes,Age on effective date,Not Applicable,"Spouse,No;Adopted Child,No;Foster Child,No;Ward,No;Stepson or Stepdaughter,No;Self,Yes;Child,No;...",16,SHOP (Small Group),Yes,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Business_Rules_PUF_Reformat.csv,2026-04-04 15:52:29.620548+00:00
5,2015,AK,73836,HIOS,15,2015-04-22 11:06:15,73836,93-0989307,73836AK073,73836AK0730004,A different rate (specifically for parties of two or more)for each enrollee is added together,3 or more,3 or more,25,3 or more,Yes,Yes,Age on effective date,Not Applicable,"Spouse,No;Adopted Child,No;Ward,No;Stepson or Stepdaughter,No;Self,Yes;Child,No;Dependent on a M...",10,SHOP (Small Group),No,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Business_Rules_PUF_Reformat.csv,2026-04-04 15:52:29.620548+00:00
6,2015,AK,73836,HIOS,15,2015-04-22 11:06:15,73836,93-0989307,73836AK073,73836AK0730005,A different rate (specifically for parties of two or more)for each enrollee is added together,3 or more,3 or more,25,3 or more,Yes,Yes,Age on effective date,Not Applicable,"Spouse,No;Adopted Child,No;Ward,No;Stepson or Stepdaughter,No;Self,Yes;Child,No;Dependent on a M...",10,SHOP (Small Group),No,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Busi

   ↳ 25 colunas | 10 linhas exibidas



---
## 8. Tabela: Crosswalk — Mapeamento de Planos entre Anos

**O que é:** Mapeia a **linhagem temporal dos planos** (2014 → 2015 → 2016). Como seguradoras podem renomear, reformular ou descontinuar planos a cada ciclo anual, o Crosswalk é o único arquivo que permite rastrear a evolução de um plano específico ao longo do tempo — peça central da arquitetura deste projeto.

No dataset existem dois arquivos: `Crosswalk2015.csv` (liga 2014 → 2015) e `Crosswalk2016.csv` (liga 2015 → 2016).

**Colunas principais:**

| Coluna | Tipo | Descrição |
|---|---|---|
| `BusinessYear` | integer | Ano de destino (ex: 2015 no Crosswalk2015) |
| `StateCode` | string | Estado |
| `IssuerId` | string | ID da seguradora no ano de destino |
| `VersionNum` | integer | Versão da submissão |
| `CrosswalkYear` | integer | Ano de origem (ex: 2014 no Crosswalk2015) |
| `CrosswalkIssuerId` | string | IssuerId do plano no ano de origem |
| `CrosswalkLevel` | string | Nível de equivalência: `Substitute`, `Renewal`, `Transition`, `New Product` |
| `CrosswalkPlanId` | string | PlanId do plano no ano de origem |
| `PlanId` | string | PlanId do plano equivalente no ano de destino |
| `CrosswalkReason` | string | Motivo da mudança (ex: mudança de rede, renomeação, fusão) |

**Como usar — rastrear um plano de 2014 ao equivalente em 2016:**
```sql
-- Passo 1: 2014 → 2015
SELECT crosswalk_plan_id AS plan_2014, plan_id AS plan_2015
FROM crosswalk_2015
WHERE crosswalk_plan_id = '12345TX1234567';

-- Passo 2: 2015 → 2016
SELECT crosswalk_plan_id AS plan_2015, plan_id AS plan_2016
FROM crosswalk_2016
WHERE crosswalk_plan_id = '<plan_2015 do passo anterior>';
```

**Relações:** `CrosswalkPlanId` → `PlanAttributes.PlanId` do ano de origem; `PlanId` → `PlanAttributes.PlanId` do ano de destino.

**Insights críticos do Kaggle:**
- ~70% dos planos de 2014 têm correspondente no Crosswalk2015 — os restantes foram descontinuados.
- Muitos planos mudam de `PlanId` entre anos **sem mudança real de produto** (apenas renomeação). Sem o Crosswalk, comparar preços ano a ano é comparar produtos diferentes.
- Planos descontinuados não aparecem como destino no Crosswalk — apenas como origem.

**Relevância para o projeto:** **Peça central da arquitetura** — sem o Crosswalk, a análise temporal de inflação médica (Questão 1) e evolução de prêmios compararia "maçãs com laranjas". O ETL de Crosswalk foi identificado como o teste unitário obrigatório do projeto (validar 100 planos que mudaram de ID entre 2015 e 2016).

In [10]:
show_family(["crosswalk"])

### `crosswalk2015`

,state,dentalplan,planid_2014,issuerid_2014,multistateplan_2014,metallevel_2014,childadultonly_2014,fipscode,zipcode,crosswalklevel,reasonforcrosswalk,planid_2015,issuerid_2015,multistateplan_2015,metallevel_2015,childadultonly_2015,ageoffplanid_2015,issuerid_ageoff2015,multistateplan_ageoff2015,metallevel_ageoff2015,childadultonly_ageoff2015,landinzone_path,ingestion_datetime
0,AK,Y,21989AK0010001,21989,N,Low,0,2013,0,1,6,21989AK0030001,21989,N,High,0,00000XX0000000,0,X,X,X,s3://637423524537-eedb015-landing/raw/health_insurance/Crosswalk2015.csv,2026-04-04 15:49:07.470327+00:00
1,AK,Y,21989AK0010001,21989,N,Low,0,2016,0,1,6,21989AK0030001,21989,N,High,0,00000XX0000000,0,X,X,X,s3://637423524537-eedb015-landing/raw/health_insurance/Crosswalk2015.csv,2026-04-04 15:49:07.470327+00:00
2,AK,Y,21989AK0010001,21989,N,Low,0,2020,0,1,6,21989AK0030001,21989,N,High,0,00000XX0000000,0,X,X,X,s3://637423524537-eedb015-landing/raw/health_insurance/Crosswalk2015.csv,2026-04-04 15:49:07.470327+00:00
3,AK,Y,21989AK0010001,21989,N,Low,0,2050,0,1,6,21989AK0030001,21989,N,High,0,00000XX0000000,0,X,X,X,s3://637423524537-eedb015-landing/raw/health_insurance/Crosswalk2015.csv,2026-04-04 15:49:07.470327+00:00
4,AK,Y,21989AK0010001,21989,N,Low,0,2060,0,1,6,21989AK0030001,21989,N,High,0,00000XX0000000,0,X,X,X,s3://637423524537-eedb015-landing/raw/health_insurance/Crosswalk2015.csv,2026-04-04 15:49:07.470327+00:00
5,AK,Y,21989AK0010001,21989,N,Low,0,2068,0,1,6,21989AK0030001,21989,N,High,0,00000XX0000000,0,X,X,X,s3://637423524537-eedb015-landing/raw/health_insurance/Crosswalk2015.csv,2026-04-04 15:49:07.470327+00:00
6,AK,Y,21989AK0010001,21989,N,Low,0,2070,0,1,6,21989AK0030001,21989,N,High,0,00000XX0000000,0,X,X,X,s3://637423524537-eedb015-landing/raw/health_insurance/Crosswalk2015.csv,2026-04-04 15:49:07.470327+00:00
7,AK,Y,21989AK0010001,21989,N,Low,0,2090,0,1,6,21989AK0030001,21989,N,High,0,00000XX0000000,0,X,X,X,s3://637423524537-eedb015-landing/raw/health_insurance/Crosswalk2015.csv,2026-04-04 15:49:07.470327+00:00
8,AK,Y,21989AK0010001,21989,N,Low,0,2100,0,1,6,21989AK0030001,21989,N,High,0,00000XX0000000,0,X,X,X,s3://637423524537-eedb015-landing/raw/health_insurance/Crosswalk2015.csv,2026-04-04 15:49:07.470327+00:00
9,AK,Y,21989AK0010001,21989,N,Low,0,2105,0,1,6,21989AK0030001,21989,N,High,0,00000XX0000000,0,X,X,X,s3://637423524537-eedb015-landing/raw/health_insurance/Crosswalk2015.csv,2026-04-04 15:49:07.470327+00:00


   ↳ 23 colunas | 10 linhas exibidas



### `crosswalk2016`

,state,dentalplan,planid_2015,issuerid_2015,multistateplan_2015,metallevel_2015,childadultonly_2015,fipscode,zipcode,crosswalklevel,reasonforcrosswalk,planid_2016,issuerid_2016,multistateplan_2016,metallevel_2016,childadultonly_2016,ageoffplanid_2016,issuerid_ageoff2016,multistateplan_ageoff2016,metallevel_ageoff2016,childadultonly_ageoff2016,landinzone_path,ingestion_datetime
0,OR,N,10091OR0360004,10091,N,Bronze,0,41021,0,0,0,10091OR0360004,10091,N,Bronze,0,00000XX0000000,0,X,X,X,s3://637423524537-eedb015-landing/raw/health_insurance/Crosswalk2016.csv,2026-04-04 15:49:10.234880+00:00
1,OR,N,10091OR0360004,10091,N,Bronze,0,41037,0,0,0,10091OR0360004,10091,N,Bronze,0,00000XX0000000,0,X,X,X,s3://637423524537-eedb015-landing/raw/health_insurance/Crosswalk2016.csv,2026-04-04 15:49:10.234880+00:00
2,OR,N,10091OR0360004,10091,N,Bronze,0,41009,0,0,0,10091OR0360004,10091,N,Bronze,0,00000XX0000000,0,X,X,X,s3://637423524537-eedb015-landing/raw/health_insurance/Crosswalk2016.csv,2026-04-04 15:49:10.234880+00:00
3,OR,N,10091OR0360004,10091,N,Bronze,0,41023,0,0,0,10091OR0360004,10091,N,Bronze,0,00000XX0000000,0,X,X,X,s3://637423524537-eedb015-landing/raw/health_insurance/Crosswalk2016.csv,2026-04-04 15:49:10.234880+00:00
4,OR,N,10091OR0360004,10091,N,Bronze,0,41059,0,0,0,10091OR0360004,10091,N,Bronze,0,00000XX0000000,0,X,X,X,s3://637423524537-eedb015-landing/raw/health_insurance/Crosswalk2016.csv,2026-04-04 15:49:10.234880+00:00
5,OR,N,10091OR0360004,10091,N,Bronze,0,41045,0,0,0,10091OR0360004,10091,N,Bronze,0,00000XX0000000,0,X,X,X,s3://637423524537-eedb015-landing/raw/health_insurance/Crosswalk2016.csv,2026-04-04 15:49:10.234880+00:00
6,OR,N,10091OR0360004,10091,N,Bronze,0,41069,0,0,0,10091OR0360004,10091,N,Bronze,0,00000XX0000000,0,X,X,X,s3://637423524537-eedb015-landing/raw/health_insurance/Crosswalk2016.csv,2026-04-04 15:49:10.234880+00:00
7,OR,N,10091OR0360004,10091,N,Bronze,0,41025,0,0,0,10091OR0360004,10091,N,Bronze,0,00000XX0000000,0,X,X,X,s3://637423524537-eedb015-landing/raw/health_insurance/Crosswalk2016.csv,2026-04-04 15:49:10.234880+00:00
8,OR,N,10091OR0360004,10091,N,Bronze,0,41063,0,0,0,10091OR0360004,10091,N,Bronze,0,00000XX0000000,0,X,X,X,s3://637423524537-eedb015-landing/raw/health_insurance/Crosswalk2016.csv,2026-04-04 15:49:10.234880+00:00
9,OR,N,10091OR0360004,10091,N,Bronze,0,41071,0,0,0,10091OR0360004,10091,N,Bronze,0,00000XX0000000,0,X,X,X,s3://637423524537-eedb015-landing/raw/health_insurance/Crosswalk2016.csv,2026-04-04 15:49:10.234880+00:00


...

,state,dentalplan,planid_2014,issuerid_2014,multistateplan_2014,metallevel_2014,childadultonly_2014,fipscode,zipcode,crosswalklevel,reasonforcrosswalk,planid_2015,issuerid_2015,multistateplan_2015,metallevel_2015,childadultonly_2015,ageoffplanid_2015,issuerid_ageoff2015,multistateplan_ageoff2015,metallevel_ageoff2015,childadultonly_ageoff2015,landinzone_path,ingestion_datetime
0,AK,Y,21989AK0010001,21989,N,Low,0,02013,00000,1,6,21989AK0030001,21989,N,High,0,00000XX0000000,00000,X,X,X,s3://637423524537-eedb015-landing/raw/health_insurance/raw/2015/Plan_Crosswalk_PUF_2014-12-22.csv,2026-04-04 15:51:01.974295+00:00
1,AK,Y,21989AK0010001,21989,N,Low,0,02016,00000,1,6,21989AK0030001,21989,N,High,0,00000XX0000000,00000,X,X,X,s3://637423524537-eedb015-landing/raw/health_insurance/raw/2015/Plan_Crosswalk_PUF_2014-12-22.csv,2026-04-04 15:51:01.974295+00:00
2,AK,Y,21989AK0010001,21989,N,Low,0,02020,00000,1,6,21989AK0030001,21989,N,High,0,00000XX0000000,00000,X,X,X,s3://637423524537-eedb015-landing/raw/health_insurance/raw/2015/Plan_Crosswalk_PUF_2014-12-22.csv,2026-04-04 15:51:01.974295+00:00
3,AK,Y,21989AK0010001,21989,N,Low,0,02050,00000,1,6,21989AK0030001,21989,N,High,0,00000XX0000000,00000,X,X,X,s3://637423524537-eedb015-landing/raw/health_insurance/raw/2015/Plan_Crosswalk_PUF_2014-12-22.csv,2026-04-04 15:51:01.974295+00:00
4,AK,Y,21989AK0010001,21989,N,Low,0,02060,00000,1,6,21989AK0030001,21989,N,High,0,00000XX0000000,00000,X,X,X,s3://637423524537-eedb015-landing/raw/health_insurance/raw/2015/Plan_Crosswalk_PUF_2014-12-22.csv,2026-04-04 15:51:01.974295+00:00
5,AK,Y,21989AK0010001,21989,N,Low,0,02068,00000,1,6,21989AK0030001,21989,N,High,0,00000XX0000000,00000,X,X,X,s3://637423524537-eedb015-landing/raw/health_insurance/raw/2015/Plan_Crosswalk_PUF_2014-12-22.csv,2026-04-04 15:51:01.974295+00:00
6,AK,Y,21989AK0010001,21989,N,Low,0,02070,00000,1,6,21989AK0030001,21989,N,High,0,00000XX0000000,00000,X,X,X,s3://637423524537-eedb015-landing/raw/health_insurance/raw/2015/Plan_Crosswalk_PUF_2014-12-22.csv,2026-04-04 15:51:01.974295+00:00
7,AK,Y,21989AK0010001,21989,N,Low,0,02090,00000,1,6,21989AK0030001,21989,N,High,0,00000XX0000000,00000,X,X,X,s3://637423524537-eedb015-landing/raw/health_insurance/raw/2015/Plan_Crosswalk_PUF_2014-12-22.csv,2026-04-04 15:51:01.974295+00:00
8,AK,Y,21989AK0010001,21989,N,Low,0,02100,00000,1,6,21989AK0030001,21989,N,High,0,00000XX0000000,00000,X,X,X,s3://637423524537-eedb015-landing/raw/health_insurance/raw/2015/Plan_Crosswalk_PUF_2014-12-22.csv,2026-04-04 15:51:01.974295+00:00
9,AK,Y,21989AK0010001,21989,N,Low,0,02105,00000,1,6,21989AK0030001,21989,N,High,0,00000XX0000000,00000,X,X,X,s3://637423524537-eedb015-landing/raw/health_insurance/raw/2015/Plan_Crosswalk_PUF_2014-12-22.csv,2026-04-04 15:51:01.974295+00:00


   ↳ 23 colunas | 10 linhas exibidas



### `raw_plan_crosswalk_puf_2014_12_22`

,state,dentalplan,planid_2014,issuerid_2014,multistateplan_2014,metallevel_2014,childadultonly_2014,fipscode,zipcode,crosswalklevel,reasonforcrosswalk,planid_2015,issuerid_2015,multistateplan_2015,metallevel_2015,childadultonly_2015,ageoffplanid_2015,issuerid_ageoff2015,multistateplan_ageoff2015,metallevel_ageoff2015,childadultonly_ageoff2015,landinzone_path,ingestion_datetime
0,AK,Y,21989AK0010001,21989,N,Low,0,02013,00000,1,6,21989AK0030001,21989,N,High,0,00000XX0000000,00000,X,X,X,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Plan_Crosswalk_PUF_2014-12-22.csv,2026-04-04 15:52:42.197388+00:00
1,AK,Y,21989AK0010001,21989,N,Low,0,02050,00000,1,6,21989AK0030001,21989,N,High,0,00000XX0000000,00000,X,X,X,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Plan_Crosswalk_PUF_2014-12-22.csv,2026-04-04 15:52:42.197388+00:00
2,AK,Y,21989AK0010001,21989,N,Low,0,02060,00000,1,6,21989AK0030001,21989,N,High,0,00000XX0000000,00000,X,X,X,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Plan_Crosswalk_PUF_2014-12-22.csv,2026-04-04 15:52:42.197388+00:00
3,AK,Y,21989AK0010001,21989,N,Low,0,02068,00000,1,6,21989AK0030001,21989,N,High,0,00000XX0000000,00000,X,X,X,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Plan_Crosswalk_PUF_2014-12-22.csv,2026-04-04 15:52:42.197388+00:00
4,AK,Y,21989AK0010001,21989,N,Low,0,02070,00000,1,6,21989AK0030001,21989,N,High,0,00000XX0000000,00000,X,X,X,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Plan_Crosswalk_PUF_2014-12-22.csv,2026-04-04 15:52:42.197388+00:00
5,AK,Y,21989AK0010001,21989,N,Low,0,02016,00000,1,6,21989AK0030001,21989,N,High,0,00000XX0000000,00000,X,X,X,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Plan_Crosswalk_PUF_2014-12-22.csv,2026-04-04 15:52:42.197388+00:00
6,AK,Y,21989AK0010001,21989,N,Low,0,02020,00000,1,6,21989AK0030001,21989,N,High,0,00000XX0000000,00000,X,X,X,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Plan_Crosswalk_PUF_2014-12-22.csv,2026-04-04 15:52:42.197388+00:00
7,AK,Y,21989AK0010001,21989,N,Low,0,02090,00000,1,6,21989AK0030001,21989,N,High,0,00000XX0000000,00000,X,X,X,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Plan_Crosswalk_PUF_2014-12-22.csv,2026-04-04 15:52:42.197388+00:00
8,AK,Y,21989AK0010001,21989,N,Low,0,02100,00000,1,6,21989AK0030001,21989,N,High,0,00000XX0000000,00000,X,X,X,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Plan_Crosswalk_PUF_2014-12-22.csv,2026-04-04 15:52:42.197388+00:00
9,AK,Y,21989AK0010001,21989,N,Low,0,02105,00000,1,6,21989AK0030001,21989,N,High,0,00000XX0000000,00000,X,X,X,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Plan_Crosswalk_PUF_2014-12-22.csv,2026-04-04 15:52:42.197388+00:00


   ↳ 23 colunas | 10 linhas exibidas



---
## 9. Tabela: Network — Rede de Prestadores

**O que é:** Relaciona cada seguradora à(s) sua(s) **rede(s) de prestadores** (hospitais, clínicas, médicos). É a menor tabela do dataset — contém apenas os metadados da rede, não a lista individual de prestadores. Múltiplos planos de uma mesma seguradora podem compartilhar o mesmo `NetworkId`.

**Colunas principais:**

| Coluna | Tipo | Descrição |
|---|---|---|
| `BusinessYear` | integer | Ano |
| `StateCode` | string | Estado |
| `IssuerId` | string | Código da seguradora |
| `SourceName` | string | Fonte/exchange |
| `VersionNum` | integer | Versão da submissão |
| `ImportDate` | date | Data de importação |
| `IssuerId2` | string | ID alternativo |
| `FederalTIN` | string | TIN federal |
| `NetworkId` | string | Identificador único da rede — chave para `PlanAttributes.NetworkId` |
| `NetworkName` | string | Nome comercial da rede (ex: "Blue Shield Trio HMO Network") |
| `NetworkURL` | string | URL do diretório público de prestadores in-network |

**Relações:** `NetworkId` → `PlanAttributes.NetworkId`; `IssuerId` → `PlanAttributes.IssuerId`.

**Limitação importante:** O dataset do Kaggle **não inclui a lista individual de prestadores** — apenas os metadados da rede. A `NetworkURL` aponta para portais externos com os diretórios de prestadores, mas estão fora do escopo do dataset. Para este projeto, o `NetworkId` serve como **proxy da amplitude da rede**: seguradoras com múltiplos `NetworkId` distintos ou nomes de rede associados a áreas geográficas menores tendem a ter redes mais restritas.

**Relevância para o projeto:** Diretamente relevante para a **Questão 4** — a relação entre amplitude da rede de prestadores e o preço final do plano.

In [11]:
show_family(["network"])

### `network`

,businessyear,statecode,issuerid,sourcename,versionnum,importdate,issuerid2,statecode2,networkname,networkid,networkurl,rownumber,marketcoverage,dentalonlyplan,landinzone_path,ingestion_datetime
0,2014,AK,21989,HIOS,6,2014-03-19 07:06:49,21989,AK,ODS Premier,AKN001,https://www.modahealth.com/ProviderSearch/faces/webpages/home.xhtml?dn=ods,13,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/Network.csv,2026-04-04 15:49:12.812739+00:00
1,2014,AK,38344,HIOS,6,2013-08-28 08:15:53,38344,AK,HeritagePlus,AKN001,https://www.premera.com/wa/visitor/,13,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/Network.csv,2026-04-04 15:49:12.812739+00:00
2,2014,AK,38536,HIOS,2,2013-08-01 12:48:00,38536,AK,Lincoln Dental Connect,AKN001,http://lfg.go2dental.com/member/dental_search/searchprov.cgi?P=LFGDentalConnect&Network=L,13,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/Network.csv,2026-04-04 15:49:12.812739+00:00
3,2014,AK,42507,HIOS,3,2013-09-02 11:39:25,42507,AK,DentalGuard Preferred,AKN001,https://www.guardiananytime.com/fpapp/FPWeb/dentalSearch.process,13,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/Network.csv,2026-04-04 15:49:12.812739+00:00
4,2014,AK,73836,HIOS,6,2014-04-18 11:49:29,73836,AK,Moda Plus AK Regional,AKN001,https://www.modahealth.com/ProviderSearch/faces/webpages/home.xhtml?dn=ods,13,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/Network.csv,2026-04-04 15:49:12.812739+00:00
5,2014,AK,73836,HIOS,6,2014-04-18 11:49:29,73836,AK,Moda Plus Providence,AKN002,https://www.modahealth.com/ProviderSearch/faces/webpages/home.xhtml?dn=ods,14,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/Network.csv,2026-04-04 15:49:12.812739+00:00
6,2014,AK,74819,HIOS,7,2014-01-21 08:29:49,74819,AK,DenteMax,AKN001,http://www2.dentemax.com/,13,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/Network.csv,2026-04-04 15:49:12.812739+00:00
7,2014,AK,84859,HIOS,1,2013-06-06 10:50:48,84859,AK,Ameritas PPO Dental Network,AKN001,www.standard.com,13,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/Network.csv,2026-04-04 15:49:12.812739+00:00
8,2014,AL,12538,HIOS,8,2014-01-21 08:29:49,12538,AL,DenteMax,ALN001,http://www2.dentemax.com/,13,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/Network.csv,2026-04-04 15:49:12.812739+00:00
9,2014,AL,12538,HIOS,8,2014-01-21 08:29:49,12538,AL,CONNECTION Dental through PPO USA,ALN002,http://www.ppousa.com/patients/index.html,14,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/Network.csv,2026-04-04 15:49:12.812739+00:00


   ↳ 16 colunas | 10 linhas exibidas



### `raw_2014_network_puf`

,businessyear,statecode,issuerid,sourcename,versionnum,importdate,issuerid2,statecode2,networkname,networkid,networkurl,rownumber,marketcoverage,dentalonly,landinzone_path,ingestion_datetime
0,2014,AK,21989,HIOS,6,2014-03-19 07:06:49,21989,AK,ODS Premier,AKN001,https://www.modahealth.com/ProviderSearch/faces/webpages/home.xhtml?dn=ods,13,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/raw/2014/Network_PUF.csv,2026-04-04 15:50:17.950874+00:00
1,2014,AK,42507,HIOS,3,2013-09-02 11:39:25,42507,AK,DentalGuard Preferred,AKN001,https://www.guardiananytime.com/fpapp/FPWeb/dentalSearch.process,13,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/raw/2014/Network_PUF.csv,2026-04-04 15:50:17.950874+00:00
2,2014,AK,73836,HIOS,6,2014-04-18 11:49:29,73836,AK,Moda Plus AK Regional,AKN001,https://www.modahealth.com/ProviderSearch/faces/webpages/home.xhtml?dn=ods,13,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/raw/2014/Network_PUF.csv,2026-04-04 15:50:17.950874+00:00
3,2014,AK,73836,HIOS,6,2014-04-18 11:49:29,73836,AK,Moda Plus Providence,AKN002,https://www.modahealth.com/ProviderSearch/faces/webpages/home.xhtml?dn=ods,14,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/raw/2014/Network_PUF.csv,2026-04-04 15:50:17.950874+00:00
4,2014,AK,74819,HIOS,7,2014-01-21 08:29:49,74819,AK,DenteMax,AKN001,http://www2.dentemax.com/,13,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/raw/2014/Network_PUF.csv,2026-04-04 15:50:17.950874+00:00
5,2014,AK,38344,HIOS,6,2013-08-28 08:15:53,38344,AK,HeritagePlus,AKN001,https://www.premera.com/wa/visitor/,13,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/raw/2014/Network_PUF.csv,2026-04-04 15:50:17.950874+00:00
6,2014,AK,38536,HIOS,2,2013-08-01 12:48:00,38536,AK,Lincoln Dental Connect,AKN001,http://lfg.go2dental.com/member/dental_search/searchprov.cgi?P=LFGDentalConnect&Network=L,13,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/raw/2014/Network_PUF.csv,2026-04-04 15:50:17.950874+00:00
7,2014,AK,84859,HIOS,1,2013-06-06 10:50:48,84859,AK,Ameritas PPO Dental Network,AKN001,www.standard.com,13,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/raw/2014/Network_PUF.csv,2026-04-04 15:50:17.950874+00:00
8,2014,AL,12538,HIOS,8,2014-01-21 08:29:49,12538,AL,DenteMax,ALN001,http://www2.dentemax.com/,13,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/raw/2014/Network_PUF.csv,2026-04-04 15:50:17.950874+00:00
9,2014,AL,12538,HIOS,8,2014-01-21 08:29:49,12538,AL,CONNECTION Dental through PPO USA,ALN002,http://www.ppousa.com/patients/index.html,14,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/raw/2014/Network_PUF.csv,2026-04-04 15:50:17.950874+00:00


...

,businessyear,statecode,issuerid,sourcename,versionnum,importdate,issuerid2,statecode2,networkname,networkid,networkurl,rownumber,marketcoverage,dentalonly,landinzone_path,ingestion_datetime
0,2015,GA,89942,HIOS,7,2015-02-19 06:21:02,89942,GA,Kaiser Permanente HMO,GAN001,kp.org/gaprovider,13,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Network_PUF.csv,2026-04-04 15:52:31.422234+00:00
1,2015,GA,93332,HIOS,10,2014-09-29 21:43:29,93332,GA,Atlanta HMOx,GAN001,https://www.humana.com/finder/search?customerId=1059&pfpkey=804,13,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Network_PUF.csv,2026-04-04 15:52:31.422234+00:00
2,2015,GA,93332,HIOS,10,2014-09-29 21:43:29,93332,GA,"Columbus, GA HMOx",GAN002,https://www.humana.com/finder/search?customerId=1059&pfpkey=758,14,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Network_PUF.csv,2026-04-04 15:52:31.422234+00:00
3,2015,GA,93332,HIOS,10,2014-09-29 21:43:29,93332,GA,Savannah HMOx,GAN003,https://www.humana.com/finder/search?customerId=1059&pfpkey=805,15,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Network_PUF.csv,2026-04-04 15:52:31.422234+00:00
4,2015,GA,93332,HIOS,10,2014-09-29 21:43:29,93332,GA,Macon HMOx,GAN005,https://www.humana.com/finder/search?customerId=1059&pfpkey=925,16,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Network_PUF.csv,2026-04-04 15:52:31.422234+00:00
5,2015,GA,93332,HIOS,10,2014-09-29 21:43:29,93332,GA,National POS - Open Access,GAN004,https://www.humana.com/finder/search?customerId=1059&pfpkey=319,17,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Network_PUF.csv,2026-04-04 15:52:31.422234+00:00
6,2015,GA,98239,HIOS,4,2014-09-06 03:39:47,98239,GA,Careington Maximum Care Network,GAN001,www.careington.com/co/maxcare,13,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Network_PUF.csv,2026-04-04 15:52:31.422234+00:00
7,2015,IN,10064,HIOS,2,2014-08-08 08:53:29,10064,IN,Principal Plan Dental,INN001,http://c3.go2dental.com/member/dental_search/provsel.cgi,13,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Network_PUF.csv,2026-04-04 15:52:31.422234+00:00
8,2015,IN,17575,HIOS,14,2014-12-12 09:23:47,17575,IN,Pathway X HMO/POS,INN001,https://www.anthem.com/health-insurance/provider-directory/searchcriteria?alphaprefix=XPH,13,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Network_PUF.csv,2026-04-04 15:52:31.422234+00:00
9,2015,IN,17575,HIOS,14,2014-12-12 09:23:47,17575,IN,Pathway X HMO/POS with Dental,INN002,https://www.anthem.com/health-insurance/provider-directory/searchcriteria?alphaprefix=XPH,14,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Network_PUF.csv,2026-04-04 15:52:31.422234+00:00


   ↳ 16 colunas | 10 linhas exibidas



### `raw_network_puf_2015_12_08`

,businessyear,statecode,issuerid,sourcename,versionnum,importdate,issuerid2,statecode2,networkname,networkid,networkurl,rownumber,marketcoverage,dentalonlyplan,landinzone_path,ingestion_datetime
0,2016,AK,21989,HIOS,4,2015-08-22 15:09:32,21989,AK,Delta Dental Premier,AKN001,https://www.modahealth.com/ProviderSearch/faces/webpages/home.xhtml,13,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Network_PUF_2015-12-08.csv,2026-04-04 15:52:33.115298+00:00
1,2016,AK,21989,HIOS,4,2015-08-22 15:09:32,21989,AK,Delta Dental PPO,AKN002,https://www.modahealth.com/ProviderSearch/faces/webpages/home.xhtml,14,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Network_PUF_2015-12-08.csv,2026-04-04 15:52:33.115298+00:00
2,2016,AK,38344,HIOS,10,2015-08-25 05:06:23,38344,AK,HeritagePlus and Dental,AKN001,https://premera.vitalschoice.com/#/?ci=premeraak&network_id=4,13,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Network_PUF_2015-12-08.csv,2026-04-04 15:52:33.115298+00:00
3,2016,AK,47904,HIOS,2,2015-07-10 02:19:03,47904,AK,Delta Dental,AKN002,http://www.deltadental.com/DentistSearch/DentistSearchController.ccl,14,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Network_PUF_2015-12-08.csv,2026-04-04 15:52:33.115298+00:00
4,2016,AK,58670,HIOS,2,2015-07-08 02:47:08,58670,AK,Indemnity,AKN001,Indemnity,13,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Network_PUF_2015-12-08.csv,2026-04-04 15:52:33.115298+00:00
5,2016,AK,73836,HIOS,4,2015-08-23 12:37:12,73836,AK,Endeavor Providence,AKN002,https://www.modahealth.com/ProviderSearch/faces/webpages/home.xhtml,13,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Network_PUF_2015-12-08.csv,2026-04-04 15:52:33.115298+00:00
6,2016,AK,38344,HIOS,10,2015-08-25 05:06:23,38344,AK,Dental,AKN002,https://premera.vitalschoice.com/#/?ci=premeraak&network_id=7,14,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Network_PUF_2015-12-08.csv,2026-04-04 15:52:33.115298+00:00
7,2016,AK,38344,HIOS,10,2015-08-25 05:06:23,38344,AK,HeritagePlus,AKN003,https://premera.vitalschoice.com/#/?ci=premeraak&network_id=4,15,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Network_PUF_2015-12-08.csv,2026-04-04 15:52:33.115298+00:00
8,2016,AK,45858,HIOS,1,2015-05-01 02:23:41,45858,AK,Ameritas PPO Dental Network,AKN001,www.ameritas.com,13,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Network_PUF_2015-12-08.csv,2026-04-04 15:52:33.115298+00:00
9,2016,AK,47904,HIOS,2,2015-07-10 02:19:03,47904,AK,Renaissance Dental,AKN001,www.renaissancedental.com/Find-a-Dentist,13,<NA>,<NA>,s3://637423524537-eedb015-landing/raw/health_insurance/raw/Network_PUF_2015-12-08.csv,2026-04-04 15:52:33.115298+00:00


   ↳ 16 colunas | 10 linhas exibidas



---
## 10. Tabelas Adicionais

Tabelas presentes no database que não se encaixam nas famílias principais acima. Podem incluir arquivos auxiliares, variantes ou tabelas de controle criadas durante a ingestão.

In [12]:
# Identifica tabelas que não foram cobertas pelas seções anteriores
known_patterns = ["rate", "plan_attributes", "benefits_cost_sharing", "benefits",
                  "service_area", "business_rules", "crosswalk", "network"]

other_tables = [
    t for t in all_table_names
    if not any(p in t for p in known_patterns)
]

if other_tables:
    print(f"Tabelas adicionais encontradas: {other_tables}")
    show_family(other_tables)
else:
    print("Não há tabelas adicionais fora das famílias mapeadas.")

Não há tabelas adicionais fora das famílias mapeadas.


---
## 11. Resumo das Relações entre Tabelas

```
Crosswalk2015 / Crosswalk2016
  CrosswalkPlanId ──► PlanId (no ano seguinte)
                            │
                            ▼
ServiceArea ──────► PlanAttributes ◄──── BenefitsCostSharing
(ServiceAreaId)      │  (PlanId)               (PlanId)
                     │
Network ─────────────┤ (NetworkId)
                     │
                     ├──── PlanId ──── Rate
                     │                  └── RatingAreaId → ServiceArea
                     │
                     └──── PlanId ──── BusinessRules
```

**Chave central:** `PlanId` no formato HIOS — `IssuerId(5) + ProductId(2) + PlanId(2) + VariantId(2)`. Liga todas as tabelas analíticas.

### Mapeamento para as Questões do Projeto

| Questão | Tabelas Primárias | Tabelas de Apoio |
|---|---|---|
| Q1 — Custos oncológicos (Copay × Coinsurance 2014–2016) | `BenefitsCostSharing` | `PlanAttributes` (MetalLevel), `Crosswalk` |
| Q2 — Competição × Preço médio por estado | `Rate`, `PlanAttributes` | `ServiceArea` |
| Q3 — Benefícios como variável de precificação | `BenefitsCostSharing`, `Rate` | `PlanAttributes` |
| Q4 — Tamanho da rede × Preço | `Network`, `Rate` | `PlanAttributes`, `ServiceArea` |
| Q5 — Monopólios e desigualdade geográfica | `ServiceArea`, `Rate` | `PlanAttributes` |
| Rastreio temporal (linhagem) | `Crosswalk2015`, `Crosswalk2016` | `PlanAttributes`, `Rate` |

---
## 12. Qualidade dos Dados — Pontos de Atenção para o ETL Silver

Com base nas análises públicas do Kaggle e na documentação do CMS, estes são os principais problemas de qualidade conhecidos que o ETL da camada Silver deve tratar:

| # | Tabela | Problema | Impacto | Tratamento sugerido |
|---|---|---|---|---|
| 1 | `BenefitsCostSharing` | Campos `CopayInnTier1/2` e `CoinsInnTier1/2` chegam como strings mistas: `"$30"`, `"$0"`, `"No Charge"`, `"Not Applicable"` | Impossível agregar sem normalização | Parsear para decimal; mapear `"No Charge"` → `0.0`, `"Not Applicable"` → `null` |
| 2 | `BenefitsCostSharing` | `CopayInnTier2` e `CoinsInnTier2` têm alta taxa de nulos — muitos planos operam apenas com Tier 1 | Nulos não são ausência de dado, são "não aplicável" | Distinguir `null` legítimo de dado faltante; não imputar |
| 3 | `PlanAttributes` | Variantes CSR do mesmo plano (sufixo `73`, `87`, etc. no `PlanId`) representam o mesmo produto com deductibles reduzidos | Joins podem duplicar linhas de preço | Agrupar variantes pelo `HIOSProductId` antes de joins com `Rate` |
| 4 | `Rate` | ~50M linhas no total — sem particionamento adequado, scans completos são proibitivos no Athena | Custo e latência altos | Particionamento por `StateCode` + `BusinessYear` (já aplicado na Bronze via Iceberg) |
| 5 | `Crosswalk` | Planos descontinuados aparecem como `CrosswalkPlanId` sem `PlanId` de destino | Left joins retornam nulos para planos extintos | Tratar nulos como descontinuações; não descartar esses registros |
| 6 | `Rate` | Valores extremos em `IndividualRate`: registros com `0.0` ou valores acima de `$5.000/mês` | Distorce médias sem filtro | Aplicar filtro de range razoável na Silver (ex: `> 0` e `< 3000`) |
| 7 | Geral | Colunas `VersionNum` e `ImportDate` — quando há múltiplas versões do mesmo arquivo, a versão mais recente substitui a anterior | Dados duplicados com versões diferentes | Deduplica pelo `PlanId` + ano mantendo o maior `VersionNum` |